# nanochat fine-tuning

In [ ]:
%pip install -qU transformers==4.57.1 tiktoken wandb

## 環境構築

In [ ]:
import os

# メモリ断片化によるCUDAのOOMを防ぐための環境変数を設定
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
# RustBPEのビルドとインストール

%pip install maturin
import os

if not os.path.exists("nanochat"):
    !git clone https://github.com/karpathy/nanochat

try:
    # Google Colabの場合
    from google.colab import userdata

    # RustBPEのビルドとインストール
    # curl --proto '=https' --tlsv1.2 -sSf https://sh.rustup.rs | sh -s -- -y && . "$HOME/.cargo/env" && maturin build --release --manifest-path nanochat/rustbpe/Cargo.toml && pip install nanochat/rustbpe/target/wheels/*.whl

    if not os.path.exists("nanochat/rustbpe/target"):
        raise FileNotFoundError("rustbpeのビルドとインストールをターミナルで実行してください。")

except ImportError:
    # ローカル環境の場合
    !maturin develop --release --manifest-path nanochat/rustbpe/Cargo.toml

import rustbpe

In [ ]:
# ログ設定

import logging as logging

if os.path.exists('debug.log'):
    os.remove('debug.log')

def custom_format(record):
    match record.levelno:
        case logging.DEBUG:
            level = '🟦'
        case logging.INFO:
            level = '🟩'
        case logging.WARNING:
            level = '🟨'
        case logging.ERROR:
            level = '🟥'
        case logging.CRITICAL:
            level = '🛑'
    return f"{level} {record.getMessage()}"

logger = logging.getLogger()

for handler in logger.handlers:
    logger.removeHandler(handler)

formatter = logging.Formatter()
formatter.format = custom_format

file_handler = logging.FileHandler('debug.log')
file_handler.setFormatter(formatter)
logger.addHandler(file_handler)

stream_handler = logging.StreamHandler()
stream_handler.setFormatter(formatter)
logger.addHandler(stream_handler)

logger.setLevel(logging.DEBUG)
logger.debug("ログを初期化")

In [ ]:
# ベースディレクトリを設定

def get_base_dir():
    """
    ベースディレクトリのパスを取得する
    デフォルトは、~/.cache/nanochat
    NANOCHAT_BASE_DIR環境変数で上書き可能

    Returns:
        str: ベースディレクトリのパス
    """
    if os.environ.get("NANOCHAT_BASE_DIR"):
        nanochat_dir = os.environ.get("NANOCHAT_BASE_DIR")
    else:
        home_dir = os.path.expanduser("~")
        cache_dir = os.path.join(home_dir, ".cache")
        nanochat_dir = os.path.join(cache_dir, "nanochat")
    os.makedirs(nanochat_dir, exist_ok=True)
    return nanochat_dir

# ベースディレクトリ
base_dir = get_base_dir()

# 訓練済みのトークナイザーのダウンロード先
os.makedirs(os.path.join(base_dir, "tokenizer"), exist_ok=True)

# 事前学習済みのモデルのダウンロード先
os.makedirs(os.path.join(base_dir, "base_checkpoints", "d20"), exist_ok=True)

logger.info(f"ベースディレクトリ: {base_dir}")

In [ ]:
# nanochat-studentsから事前学習済みモデルをダウンロード
# https://huggingface.co/nanochat-students

from filelock import FileLock
import urllib

BASE_URL = "https://huggingface.co/nanochat-students/base-d20/resolve/main"

def download_file_with_lock(url, filename, postprocess_fn=None):
    """
    URLからベースディレクトリのローカルパスにファイルをダウンロードする
    複数のデバイスを並列実行する場合に競合を防ぐためロックファイルを使用する

    引数:
        url (str): ダウンロード元のURL
        filename (str): ベースディレクトリからの相対パス
        postprocess_fn (function, optional): ダウンロード後に実行する関数。引数としてローカルファイルパスを受け取る
    戻り値:
        str: ローカルファイルパス
    """

    # 1) パスの準備

    base_dir = get_base_dir()
    file_path = os.path.join(base_dir, filename)
    lock_path = file_path + ".lock"

    # 2) ファイルが既に存在する場合はスキップ

    if os.path.exists(file_path):
        return file_path

    # 3) ロックを取得してダウンロード

    # 分散学習（DDP）で複数のプロセスが同時にダウンロードしないようにロックを使用
    with FileLock(lock_path):

        # 別のプロセスがダウンロードした場合はスキップ
        if os.path.exists(file_path):
            return file_path

        # バイト列としてコンテンツをダウンロード
        logger.debug(f"Downloading {url}...")
        with urllib.request.urlopen(url) as response:
            content = response.read()

        # バイト列としてローカルファイルに書き込む
        with open(file_path, 'wb') as f:
            f.write(content)

        logger.debug(f"Downloaded to {file_path}")

        # 後処理が必要な場合は実行
        if postprocess_fn is not None:
            postprocess_fn(file_path)

    return file_path

# BPB評価用のトークンとバイト数の対応リスト
download_file_with_lock(f"{BASE_URL}/token_bytes.pt", "tokenizer/token_bytes.pt")

# 訓練済みのトークナイザー
download_file_with_lock(f"{BASE_URL}/tokenizer.pkl", "tokenizer/tokenizer.pkl")

# 事前学習済みのモデルチェックポイント
download_file_with_lock(f"{BASE_URL}/model_021400.pt", "base_checkpoints/d20/model_021400.pt")

# 最適化関数の状態
download_file_with_lock(f"{BASE_URL}/optim_021400.pt", "base_checkpoints/d20/optim_021400.pt")

# メタデータ
download_file_with_lock(f"{BASE_URL}/meta_021400.json", "base_checkpoints/d20/meta_021400.json")

In [ ]:
# メタデータを確認
!cat {base_dir}/base_checkpoints/d20/meta_021400.json

### トークナイザーのロード

In [ ]:
import pickle
import rustbpe
import tiktoken
import os
import copy
from functools import lru_cache

In [ ]:
# 特殊トークン

SPECIAL_TOKENS = [
    # 事前学習で使用
    "<|bos|>", # 文の開始

    # ファインチューニング時に使用
    "<|user_start|>", # ユーザーメッセージ
    "<|user_end|>",
    "<|assistant_start|>", # アシスタントメッセージ
    "<|assistant_end|>",
    "<|python_start|>", # アシスタントがPython REPLツールを呼び出す
    "<|python_end|>",
    "<|output_start|>", # Python REPLがアシスタントに出力を返す
    "<|output_end|>",
]

logger.debug(f"特殊トークン数 {len(SPECIAL_TOKENS)=}")

In [ ]:
class RustBPETokenizer:
    """
    rustbpeのラッパークラス
    訓練時はrustbpeを使用し、推論時はtiktokenを使用する
    """

    def __init__(self, enc, bos_token):
        logger.debug(f"RustBPETokenizer初期化開始 {enc=} {bos_token=}")

        # tiktokenのEncodingオブジェクト
        self.enc = enc 

        # BOSトークンIDをキャッシュ
        self.bos_token_id = self.encode_special(bos_token)
        logger.debug(f"RustBPETokenizer初期化完了 {self.bos_token_id=}")

    @classmethod # クラスメソッドとして定義
    def train_from_iterator(cls, text_iterator, vocab_size):
        """
        rustbpeでトークナイザーを訓練し、tiktokenのエンコーダーを構築する

        Args:
            text_iterator (iterable): テキストのイテレータ
            vocab_size (int): 語彙数
        Returns:
            RustBPETokenizer: 訓練済みのRustBPETokenizerオブジェクト
        """

        logger.debug(f"RustBPETokenizerの訓練開始 {vocab_size=}")

        # 1) rustbpeを訓練

        # rustbpeのTokenizerオブジェクトを作成
        tokenizer = rustbpe.Tokenizer()

        # 特殊トークンは後で__init__で挿入されるため、ここでは訓練しない
        vocab_size_no_special = vocab_size - len(SPECIAL_TOKENS)

        assert vocab_size_no_special >= 256, f"vocab_size_no_special must be at least 256, got {vocab_size_no_special}"

        # トークナイザーを訓練
        tokenizer.train_from_iterator(text_iterator, vocab_size_no_special, pattern=SPLIT_PATTERN)

        # 2) tiktokenのエンコーダーを構築

        # 事前トークン化の正規表現パターン
        pattern = tokenizer.get_pattern()

        # tiktokenに対応したマージルールを作成
        # {バイト列: マージの優先順位ランク}
        mergeable_ranks_list = tokenizer.get_mergeable_ranks()
        mergeable_ranks = {bytes(k): v for k, v in mergeable_ranks_list}

        # tiktokenに対応した特殊トークンの辞書を作成
        # {トークン名: トークンID}
        tokens_offset = len(mergeable_ranks)
        special_tokens = {name: tokens_offset + i for i, name in enumerate(SPECIAL_TOKENS)}

        # tiktokenのエンコーダーを構築
        enc = tiktoken.Encoding(
            name="rustbpe",
            pat_str=pattern,
            mergeable_ranks=mergeable_ranks,
            special_tokens=special_tokens,
        )

        logger.debug(f"RustBPETokenizerの訓練完了")

        # RustBPETokenizerオブジェクトを返す
        # clsでこのクラスのコンストラクタが呼ばれる
        return cls(enc, "<|bos|>")

    @classmethod
    def from_directory(cls, tokenizer_dir):
        """
        訓練済みのRustBPETokenizerをディレクトリから読み込む

        Args:
            tokenizer_dir (str): トークナイザーの保存ディレクトリ
        Returns:
            RustBPETokenizer: 読み込んだRustBPETokenizerオブジェクト
        """

        logger.debug(f"RustBPETokenizerをディレクトリから読み込み開始 {tokenizer_dir=}")

        pickle_path = os.path.join(tokenizer_dir, "tokenizer.pkl")
        logger.debug(f"トークナイザーファイルパス {pickle_path=}")

        with open(pickle_path, "rb") as f:
            enc = pickle.load(f)

        logger.debug(f"RustBPETokenizerをディレクトリから読み込み完了 {enc=}")
        return cls(enc, "<|bos|>")

    @classmethod
    def from_pretrained(cls, tiktoken_name):
        """
        学習済みのtiktokenトークナイザーを読み込む

        Args:
            tiktoken_name (str): tiktokenのエンコーディング名
        Returns:
            RustBPETokenizer: 読み込んだRustBPETokenizerオブジェクト
        """
        logger.debug(f"学習済みのtiktokenトークナイザーを読み込み開始 {tiktoken_name=}")

        # https://github.com/openai/tiktoken/blob/eedc8563/tiktoken_ext/openai_public.py
        enc = tiktoken.get_encoding(tiktoken_name)

        # nanochatでは<|bos|>を使用するが、tiktokenのgpt2などでは<|endoftext|>が使用されているため

        logger.debug(f"学習済みのtiktokenトークナイザーを読み込み完了 {enc=}")
        return cls(enc, "<|endoftext|>")

    def get_vocab_size(self):
        return self.enc.n_vocab

    def get_special_tokens(self):
        return self.enc.special_tokens_set

    def id_to_token(self, id):
        return self.enc.decode([id])

    @lru_cache(maxsize=32)
    def encode_special(self, text):
        return self.enc.encode_single_token(text)

    def get_bos_token_id(self):
        return self.bos_token_id

    def encode(self, text, prepend=None, append=None, num_threads=8):
        """
        テキストをトークンIDにエンコードする

        Args:
            text (str or list[str]): エンコードするテキストまたはテキストのリスト
            prepend (str or int, optional): 先頭に追加する特殊トークン（文字列またはトークンID）
            append (str or int, optional): 末尾に追加する特殊トークン（文字列またはトークンID）
            num_threads (int, optional): バッチエンコード時のスレッド数（デフォルト: 8）
        Returns:
            list[int] or list[list[int]]: トークンIDのリストまたはトークンIDのリストのリスト
        """
        logger.debug(f"エンコード開始 {len(text)=} {prepend=} {append=} {num_threads=}")

        # 1) 特殊トークンの設定

        # 出力の先頭に付ける特殊トークンIDを取得
        # BOSトークン
        if prepend is not None:
            prepend_id = prepend if isinstance(prepend, int) else self.encode_special(prepend)
            logger.debug(f"{prepend_id=}") # 65527

        # 出力の末尾に付ける特殊トークンIDを取得
        # なし
        if append is not None:
            append_id = append if isinstance(append, int) else self.encode_special(append)
            logger.debug(f"{append_id=}") 

        # 2) エンコード

        # 入力が文字列の場合
        if isinstance(text, str):

            # tiktokenでエンコード
            ids = self.enc.encode_ordinary(text)

            if prepend is not None:
                # 先頭に特殊トークンを追加
                ids.insert(0, prepend_id)

            if append is not None:
                # 末尾に特殊トークンを追加
                ids.append(append_id)

        # 入力が文字列のリストの場合
        elif isinstance(text, list):

            # tiktokenでバッチエンコード
            ids = self.enc.encode_ordinary_batch(text, num_threads=num_threads)

            if prepend is not None:
                # 先頭に特殊トークンを追加
                for ids_row in ids:
                    ids_row.insert(0, prepend_id)

            if append is not None:
                # 末尾に特殊トークンを追加
                for ids_row in ids:
                    ids_row.append(append_id)
        else:
            raise ValueError(f"Invalid input type: {type(text)}")

        logger.debug(f"エンコード完了 {len(ids)=}")
        return ids

    def __call__(self, *args, **kwargs):
        return self.encode(*args, **kwargs)

    def decode(self, ids):
        """
        トークンIDのリストをデコードしてテキストに変換する

        Args:
            ids (list[int]): トークンIDのリスト
        Returns:
            str: デコードしたテキスト
        """
        logger.debug(f"デコード開始 {len(ids)=}")

        # tiktokenでデコード
        res = self.enc.decode(ids)

        logger.debug(f"デコード完了 {len(res)=}")
        return res

    def save(self, tokenizer_dir):
        """
        訓練後のトークナイザーをディレクトリに保存する

        Args:
            tokenizer_dir (str): トークナイザーの保存ディレクトリ
        """
        logger.debug(f"トークナイザーの保存開始 {tokenizer_dir=}")

        # 1) ディレクトリを作成

        os.makedirs(tokenizer_dir, exist_ok=True)
        pickle_path = os.path.join(tokenizer_dir, "tokenizer.pkl")
        logger.debug(f"トークナイザーファイルパス {pickle_path=}")

        # 2) pickleで保存

        with open(pickle_path, "wb") as f:
            pickle.dump(self.enc, f)

        logger.debug(f"トークナイザーの保存完了 {pickle_path=}")

    def render_conversation(self, conversation, max_tokens=2048):
        """
        チャット形式のデータをトークン化する
        中間学習とSFTで使用

        Args:
            conversation: dict 対話データ
            max_tokens: int 最大トークン数
        Returns:
            ids: list[int] トークンIDのリスト
            mask: list[int] 同じ長さのマスクリスト、mask=1はアシスタントが学習すべきトークン
        """
        logger.debug(f"会話のレンダリング開始 {conversation=} {max_tokens=}")

        # 1) ヘルパー関数の定義と初期化

        # ids, masks that we will return and a helper function to help build them up.
        # ids: 最終的にモデルに入力されるトークンIDのリスト
        # mask: 損失の計算に含めるかを示すマスク（mask=1は含める、mask=0は含めない）
        ids, mask = [], []

        def add_tokens(token_ids, mask_val):
            """
            トークンを追加する際、同時にそのトークンに対応するマスク値も追加する

            Args:
                token_ids: list[int] または int 追加するトークンIDまたはトークンIDのリスト
                mask_val: int 追加するマスク値（0または1）
            Returns:
                None
            """
            # リストに正規化
            if isinstance(token_ids, int):
                token_ids = [token_ids]

            # idsにトークンIDを追加
            ids.extend(token_ids)

            # maskに対応するマスク値を追加
            mask.extend([mask_val] * len(token_ids))

        # 2) システムメッセージの処理

        # 最初がシステムメッセージの場合
        if conversation["messages"][0]["role"] == "system":
            # 単純のため、最初のシステムメッセージを2番目のユーザーメッセージにマージ
            conversation = copy.deepcopy(conversation)
            messages = conversation["messages"]
            assert messages[1]["role"] == "user", "System message must be followed by a user message"
            messages[1]["content"] = messages[0]["content"] + "\n\n" + messages[1]["content"]
            messages = messages[1:]
        else:
            messages = conversation["messages"]

        assert len(messages) >= 1, f"Conversation has less than 1 message: {messages}"

        # 3) 特殊トークンの準備

        # BOSトークン（Begin Of Sentence）
        bos = self.get_bos_token_id()

        # ユーザーメッセージの開始と終了トークン
        user_start, user_end = self.encode_special("<|user_start|>"), self.encode_special("<|user_end|>")

        # アシスタントメッセージの開始と終了トークン
        assistant_start, assistant_end = self.encode_special("<|assistant_start|>"), self.encode_special("<|assistant_end|>")

        # Pythonツール呼び出しの開始と終了トークン
        python_start, python_end = self.encode_special("<|python_start|>"), self.encode_special("<|python_end|>")

        # Python出力の開始と終了トークン
        output_start, output_end = self.encode_special("<|output_start|>"), self.encode_special("<|output_end|>")

        # 4) 文頭トークンの追加

        # 開始トークンを追加（損失計算には含めないのでマスク値は0）
        add_tokens(bos, 0)

        # 5) メッセージを順番に処理

        for i, message in enumerate(messages):

            # 5-1) 検証

            must_be_from = "user" if i % 2 == 0 else "assistant"

            assert message["role"] == must_be_from, f"Message {i} is from {message['role']} but should be from {must_be_from}"

            # 5-2) コンテンツを取得
            content = message["content"]

            # 5-3) ユーザーメッセージの場合
            if message["role"] == "user":
                assert isinstance(content, str), "User messages are simply expected to be strings"

                # コンテンツをトークン化
                value_ids = self.encode(content)

                # ユーザーメッセージのトークンを追加（損失計算には含めないのでマスク値は0）
                add_tokens(user_start, 0)
                add_tokens(value_ids, 0)
                add_tokens(user_end, 0)

            # 5-4) アシスタントメッセージの場合
            elif message["role"] == "assistant":

                # アシスタントの開始トークンを追加（損失計算には含めないのでマスク値は0）
                add_tokens(assistant_start, 0)

                # コンテンツが文字列の場合
                if isinstance(content, str):
                    # コンテンツをトークン化
                    value_ids = self.encode(content)

                    # アシスタントメッセージのトークンを追加（損失計算に含めるのでマスク値は1）
                    add_tokens(value_ids, 1)

                elif isinstance(content, list):
                    for part in content:
                        # コンテンツをトークン化
                        value_ids = self.encode(part["text"])

                        # 通常のテキストの場合
                        if part["type"] == "text":

                            # アシスタントメッセージのトークンを追加（損失計算に含めるのでマスク値は1）
                            add_tokens(value_ids, 1)

                        # Pythonのツール呼び出しの場合
                        elif part["type"] == "python":
                            # ツールの呼び出しのトークンを追加（損失計算に含めるのでマスク値は1）
                            add_tokens(python_start, 1)
                            add_tokens(value_ids, 1)
                            add_tokens(python_end, 1)

                        # Pythonの出力の場合
                        elif part["type"] == "python_output":
                            # ツールの出力のトークンを追加
                            # 推論時はPythonの出力を用いるため損失計算に含めない
                            add_tokens(output_start, 0)
                            add_tokens(value_ids, 0)
                            add_tokens(output_end, 0)
                        else:
                            raise ValueError(f"Unknown part type: {part['type']}")
                else:
                    raise ValueError(f"Unknown content type: {type(content)}")

                # 終了トークンを追加（損失計算に含めるのでマスク値は1）
                add_tokens(assistant_end, 1)

        # 最大トークン数に切り詰める
        ids = ids[:max_tokens]
        mask = mask[:max_tokens]

        logger.debug(f"会話のレンダリング完了 {ids=} {mask=}")
        return ids, mask

    def visualize_tokenization(self, ids, mask, with_token_id=False):
        """
        今回は使わない
        """
        RED = '\033[91m'
        GREEN = '\033[92m'
        RESET = '\033[0m'
        GRAY = '\033[90m'
        tokens = []
        for i, (token_id, mask_val) in enumerate(zip(ids, mask)):
            token_str = self.decode([token_id])
            color = GREEN if mask_val == 1 else RED
            tokens.append(f"{color}{token_str}{RESET}")
            if with_token_id:
                tokens.append(f"{GRAY}({token_id}){RESET}")
        return '|'.join(tokens)

    def render_for_completion(self, conversation):
        """
        会話データをトークン化して、アシスタントの応答生成用に整形する
        正解データを隠してモデルに答えさせる
        強化学習用に使用

        Args:
            conversation: dict 対話データ
        Returns:
            ids: list[int] トークンIDのリスト
        """
        # 1) データの複製と正解の削除

        conversation = copy.deepcopy(conversation)
        messages = conversation["messages"]
        assert messages[-1]["role"] == "assistant", "Last message must be from the Assistant"

        # 最後のアシスタントメッセージ（正解）を削除
        messages.pop()

        # 2) 会話データをトークン化

        ids, mask = self.render_conversation(conversation)

        # 3) アシスタントの開始トークンを追加

        assistant_start = self.encode_special("<|assistant_start|>")
        ids.append(assistant_start)

        return ids

In [ ]:
# 事前トークン化の正規表現を設定

# GPT-4と同じ
SPLIT_PATTERN = r"""'(?i:[sdmt]|ll|ve|re)|[^\r\n\p{L}\p{N}]?+\p{L}+|\p{N}{1,2}| ?[^\s\p{L}\p{N}]++[\r\n]*|\s*[\r\n]|\s+(?!\S)|\s+"""

logger.debug(f"事前トークン化の正規表現: {SPLIT_PATTERN=}")

In [ ]:
def get_tokenizer():
    """
    RustBPETokenizerオブジェクトを返す

    Returns:
        RustBPETokenizer: トークナイザーオブジェクト
    """
    base_dir = get_base_dir()
    tokenizer_dir = os.path.join(base_dir, "tokenizer")
    # return HuggingFaceTokenizer.from_directory(tokenizer_dir)
    return RustBPETokenizer.from_directory(tokenizer_dir)

tokenizer = get_tokenizer()

### モデルの実装

In [ ]:
import math
from functools import partial
from dataclasses import dataclass
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
def is_ddp():
    """
    分散データ並列（Distributed Data Parallel, DDP）が有効かどうか

    Returns:
        bool: DDPが有効な場合True、そうでない場合False
    """

    # RANK環境変数はtorchrunなどによって設定されるプロセスの識別子
    return int(os.environ.get('RANK', -1)) != -1

is_ddp()

In [ ]:
def get_dist_info():
    """
    分散データ並列（Distributed Data Parallel, DDP）訓練の環境変数を取得
    """
    if is_ddp():
        assert all(var in os.environ for var in ['RANK', 'LOCAL_RANK', 'WORLD_SIZE'])

        # グローバルなプロセス識別子
        ddp_rank = int(os.environ['RANK'])

        # ローカル（同一ノード内）のプロセス識別子
        ddp_local_rank = int(os.environ['LOCAL_RANK'])

        # ワールドサイズ（全プロセス数）
        ddp_world_size = int(os.environ['WORLD_SIZE'])
        return True, ddp_rank, ddp_local_rank, ddp_world_size
    else:
        return False, 0, 0, 1

get_dist_info()

In [ ]:
def norm(x):
    """
    平方根平均二乗ノルム正規化（Root Mean Square Normalization, RMSNorm）を適用する

    Args:
        x (Tensor): 入力テンソル
    Returns:
        Tensor: 正規化されたテンソル
    """
    logger.debug(f"RMSNormを適用開始 {x.shape=}")

    # ゲインやバイアスを持たないため高速
    # (4, 2048, 1280) -> (4, 2048, 1280)
    res = F.rms_norm(x, (x.size(-1),))

    logger.debug(f"RMSNorm適用完了 {res.shape=}")
    return res

In [ ]:
def apply_rotary_emb(x, cos, sin):
    """
    回転位置埋め込み（RoPE, Rotary Positional Embedding）を適用する

    Args:
        x (Tensor): 入力テンソル
        cos (Tensor): コサイン成分、形状は(1, seq_len, 1, head_dim/2)
        sin (Tensor): サイン成分、形状は(1, seq_len, 1, head_dim/2)
    Returns:
        Tensor: RoPEが適用されたテンソル、形状は(batch_size, seq_len, num_heads, head_dim)
    """
    logger.debug(f"RoPEを適用 {x.shape=}, {x.dtype=} {cos.shape=}, {cos.dtype=} {sin.shape=} {sin.dtype=}")

    assert x.ndim == 4

    # head_dimを2で分割
    d = x.shape[3] // 2

    # 最後の次元を2つに分割
    x1, x2 = x[..., :d], x[..., d:]

    # 2次元回転行列を適用
    y1 = x1 * cos + x2 * sin
    y2 = x1 * (-sin) + x2 * cos
    out = torch.cat([y1, y2], 3) # 最後の次元で結合

    # sinとcosはfloat32で計算されることが多いため、元のデータ型に戻す
    out = out.to(x.dtype)

    logger.debug(f"RoPE適用完了 {out.shape=}")
    return out

In [ ]:
@dataclass
class GPTConfig:
    sequence_len: int = 1024 # 最大シーケンス長
    vocab_size: int = 50304 # 語彙サイズ
    n_layer: int = 12 # Transformerのレイヤー数
    n_head: int = 6 # アテンションヘッド数
    n_kv_head: int = 6 # キーバリューヘッド数
    n_embd: int = 768 # Transformerの埋め込み次元数

In [ ]:
class CausalSelfAttention(nn.Module):
    """
    GQA（Group Query Attention）を実装した因果セルフアテンションモジュール
    """

    def __init__(self, config, layer_idx):
        logger.debug(f"CausalSelfAttentionを初期化開始 {config.n_head=} {config.n_kv_head=} {config.n_embd=} {layer_idx=}")

        super().__init__()

        self.layer_idx = layer_idx

        # 10
        self.n_head = config.n_head

        # 10
        self.n_kv_head = config.n_kv_head

        # 1280
        self.n_embd = config.n_embd

        # 1280 / 10 = 128
        self.head_dim = self.n_embd // self.n_head
        logger.debug(f"{self.head_dim=}")

        assert self.n_embd % self.n_head == 0
        assert self.n_kv_head <= self.n_head and self.n_head % self.n_kv_head == 0

        # 1280 -> 1280
        self.c_q = nn.Linear(self.n_embd, self.n_head * self.head_dim, bias=False)

        # 1280 -> 1280
        self.c_k = nn.Linear(self.n_embd, self.n_kv_head * self.head_dim, bias=False)

        # 1280 -> 1280
        self.c_v = nn.Linear(self.n_embd, self.n_kv_head * self.head_dim, bias=False)

        # 1280 -> 1280
        self.c_proj = nn.Linear(self.n_embd, self.n_embd, bias=False)

        logger.debug("CausalSelfAttentionの初期化完了")

    def forward(self, x, cos_sin, kv_cache):
        logger.debug(f"CausalSelfAttentionの順伝搬を実行 {x.shape=} {x.dtype=} {cos_sin[0].shape=} {cos_sin[0].dtype=} {kv_cache if kv_cache is not None else None=}")

        # (4, 2048, 1280)
        B, T, C = x.size()

        # 1) クエリ・キー・バリューを計算

        # (4, 2048, 1280) -> (4, 2048, 1280) -> (4, 2048, 10, 128)
        q = self.c_q(x).view(B, T, self.n_head, self.head_dim)

        # (4, 2048, 1280) -> (4, 2048, 1280) -> (4, 2048, 10, 128)
        k = self.c_k(x).view(B, T, self.n_kv_head, self.head_dim)

        # (4, 2048, 1280) -> (4, 2048, 1280) -> (4, 2048, 10, 128)
        v = self.c_v(x).view(B, T, self.n_kv_head, self.head_dim)

        # 2) クエリとキーにRoPEを適用

        # (1, 2048, 1, 64), (1, 2048, 1, 64)
        cos, sin = cos_sin

        # (4, 2048, 10, 128), (4, 2048, 10, 128)
        q, k = apply_rotary_emb(q, cos, sin), apply_rotary_emb(k, cos, sin)

        # 3) RMSNormを適用

        # (4, 2048, 10, 128) -> (4, 2048, 10, 128)
        # (4, 2048, 10, 128) -> (4, 2048, 10, 128)
        q, k = norm(q), norm(k)

        # 4) KVキャッシュを適用

        # (4, 2048, 10, 128) -> (4, 10, 2048, 128)
        # (4, 2048, 10, 128) -> (4, 10, 2048, 128)
        # (4, 2048, 10, 128) -> (4, 10, 2048, 128)
        q, k, v = q.transpose(1, 2), k.transpose(1, 2), v.transpose(1, 2)

        # KVキャッシュがある場合（推論時）
        if kv_cache is not None:
            # KVキャッシュを更新し、これまでの全てのKVを取得
            k, v = kv_cache.insert_kv(self.layer_idx, k, v)
            logger.debug(f"KVキャッシュを更新 {k.shape=} {v.shape=}")

        # 5) スケールド・ドットプロダクト・アテンションを計算

        # クエリの数
        # 2048
        Tq = q.size(2)
        logger.debug(f"{Tq=}")

        # キー/バリューの数（キャッシュ内も含む）
        # 2048
        Tk = k.size(2)
        logger.debug(f"{Tk=}")

        # GQAを有効にするかどうか
        # False
        enable_gqa = self.n_head != self.n_kv_head
        logger.debug(f"GQAを有効化: {enable_gqa=}")

        # 訓練時またはクエリ数とキー/バリュー数が等しい場合
        if kv_cache is None or Tq == Tk:
            logger.debug("通常の因果アテンションを適用")
            y = F.scaled_dot_product_attention(q, k, v, is_causal=True, enable_gqa=enable_gqa)

        # 推論時でクエリが1つだけの場合
        elif Tq == 1:
            logger.debug("マスクを使用しない因果アテンションを適用")
            y = F.scaled_dot_product_attention(q, k, v, is_causal=False, enable_gqa=enable_gqa)

        # 推論時で複数のトークンを一度に処理する場合（prefill）
        else:
            logger.debug("マスクを手動で作成して因果アテンションを適用")

            # アテンションマスクを初期化
            attn_mask = torch.zeros((Tq, Tk), dtype=torch.bool, device=q.device)
            logger.debug(f"{attn_mask.shape=}")

            # キャッシュ済みの長さを取得
            prefix_len = Tk - Tq
            logger.debug(f"{prefix_len=}")

            if prefix_len > 0:
                # マスクの左側（キャッシュ部分）をマスクしない
                attn_mask[:, :prefix_len] = True

            # マスクの右側に下三角行列を設定
            attn_mask[:, prefix_len:] = torch.tril(
                torch.ones((Tq, Tq), dtype=torch.bool, device=q.device)
            )

            y = F.scaled_dot_product_attention(q, k, v, attn_mask=attn_mask, enable_gqa=enable_gqa)

        # 6) 出力処理

        # (4, 10, 2048, 128) -> (4, 2048, 10, 128) -> (4, 2048, 1280)
        y = y.transpose(1, 2).contiguous().view(B, T, -1)
        logger.debug(f"アテンション出力を整形 {y.shape=}")

        # (4, 2048, 1280) -> (4, 2048, 1280)
        y = self.c_proj(y)
        logger.debug(f"最終線形変換を適用 {y.shape=}")

        logger.debug(f"CausalSelfAttentionの順伝搬完了 {y.shape=}")
        return y

In [ ]:
class MLP(nn.Module):
    """
    フィードフォワードネットワーク（FFN）モジュール
    """

    def __init__(self, config):
        logger.debug(f"MLPを初期化 {config.n_embd=}")
        super().__init__()

        # 1280 -> 5120
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=False)

        # 5120 -> 1280
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=False)

        logger.debug("MLPの初期化完了")

    def forward(self, x):
        logger.debug(f"MLPの順伝搬を実行 {x.shape=} {x.dtype=}")

        x = self.c_fc(x)
        logger.debug(f"{x.shape=}")

        # Squared ReLU活性化関数を適用
        # GELUやSiLU（Swish）よりも高速でメモリ効率が良い
        x = F.relu(x).square()

        x = self.c_proj(x)
        logger.debug(f"MLPの順伝搬完了 {x.shape=}")
        return x

In [ ]:
class Block(nn.Module):
    """
    Transformerブロックモジュール
    """

    def __init__(self, config, layer_idx):
        logger.debug(f"Transformer Blockを初期化 {config.n_embd=} {layer_idx=}")
        super().__init__()
        self.attn = CausalSelfAttention(config, layer_idx)
        self.mlp = MLP(config)
        logger.debug("Transformer Blockの初期化完了")

    def forward(self, x, cos_sin, kv_cache):
        logger.debug(f"Transformer Blockの順伝搬を実行 {x.shape=} {x.dtype=} {cos_sin[0].shape=} {cos_sin[0].dtype=} {kv_cache if kv_cache is not None else None=}")

        x = x + self.attn(norm(x), cos_sin, kv_cache)

        x = x + self.mlp(norm(x))

        logger.debug(f"Transformer Blockの順伝搬完了 {x.shape=}")
        return x

In [ ]:
class GPT(nn.Module):
    """
    GPTモデル全体
    """

    def __init__(self, config):
        logger.debug(f"GPTモデルを初期化 {config.n_layer=} {config.n_head=} {config.n_embd=} {config.vocab_size=} {config.sequence_len=}")

        super().__init__()

        self.config = config

        # Transformerのエンコーダー部分
        self.transformer = nn.ModuleDict({
            # 単語埋め込み層
            # 65536 -> 1280
            "wte": nn.Embedding(config.vocab_size, config.n_embd),

            # 20層のTransformerブロックを作成
            "h": nn.ModuleList(
                [Block(config, layer_idx) for layer_idx in range(config.n_layer)]
            ),
        })

        # 出力の線形層（Language Model Head）
        # 1280 -> 65536
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)

        # RoPEのシーケンス長を設定
        # KVキャッシュでシーケンスの最大長を超える可能性があるため余裕をもたせる
        # 2048 * 10 = 20480
        self.rotary_seq_len = config.sequence_len * 10
        logger.debug(f"{self.rotary_seq_len=}")

        # ヘッド次元数を計算
        # 1280 / 10 = 128
        head_dim = config.n_embd // config.n_head
        logger.debug(f"{head_dim=}")

        # RoPEのcosとsinを事前計算
        cos, sin = self._precompute_rotary_embeddings(self.rotary_seq_len, head_dim)

        # cosをバッファに登録（state_dictには保存しない）
        self.register_buffer("cos", cos, persistent=False)

        # sinをバッファに登録（state_dictには保存しない）
        self.register_buffer("sin", sin, persistent=False)

        logger.debug("GPTモデルの初期化完了")

    def init_weights(self):
        """
        モデルの重みを初期化する
        """
        logger.debug("GPTモデルの重みを初期化開始")

        # 全てのモジュールに対して初期化関数を適用
        self.apply(self._init_weights)

        # 出力層の重みをゼロに初期化
        torch.nn.init.zeros_(self.lm_head.weight)

        # 全てのTransformerブロックの出力層の重みをゼロに初期化
        for block in self.transformer.h:
            torch.nn.init.zeros_(block.mlp.c_proj.weight)
            torch.nn.init.zeros_(block.attn.c_proj.weight)

        # RoPEのcosとsinを計算
        head_dim = self.config.n_embd // self.config.n_head
        cos, sin = self._precompute_rotary_embeddings(self.rotary_seq_len, head_dim)
        self.cos, self.sin = cos, sin

        # 埋め込み層の重みをbfloat16にダウンキャストし、メモリ使用量を削減
        if self.transformer.wte.weight.device.type == "cuda":
            self.transformer.wte.to(dtype=torch.bfloat16)

        logger.debug("GPTモデルの重みの初期化完了")

    def _init_weights(self, module):
        """
        init_weightsから呼び出される重み初期化関数

        Args:
            module (nn.Module): 初期化するモジュール
        """
        logger.debug(f"モジュールの重みを初期化開始 {module.__class__.__name__=}")

        # 全結合層の場合
        if isinstance(module, nn.Linear):
            # 論文に基づく方法で重みを初期化
            # https://arxiv.org/pdf/2310.17813
            fan_out = module.weight.size(0)
            fan_in = module.weight.size(1)
            std = 1.0 / math.sqrt(fan_in) * min(1.0, math.sqrt(fan_out / fan_in))
            torch.nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)

        # 埋め込み層の場合
        elif isinstance(module, nn.Embedding):
            # 標準正規分布で重みを初期化
            torch.nn.init.normal_(module.weight, mean=0.0, std=1.0)

        logger.debug("モジュールの重みの初期化完了")

    def _precompute_rotary_embeddings(self, seq_len, head_dim, base=10000, device=None):
        """
        RoPEのcosとsinを事前計算する

        Args:
            seq_len (int): シーケンス長
            head_dim (int): ヘッド次元数
            base (int, optional): 逆周波数の基底値。デフォルトは10000。
            device (torch.device, optional): 計算に使用するデバイス。デフォルトはNoneで、自動検出される。
        Returns:
            Tuple[Tensor, Tensor]: 事前計算されたcosとsinのテンソル
        """

        logger.debug(f"RoPEの事前計算を実行 {seq_len=} {head_dim=} {base=}")

        # デバイスが指定されていない場合
        if device is None:
            # 自動検出
            device = self.transformer.wte.weight.device

        # 要素数を半分にするための偶数インデックスを作成
        channel_range = torch.arange(0, head_dim, 2, dtype=torch.float32, device=device)

        # 逆周波数（invert frequency）を計算
        # 角速度と実質的に同じ
        # \theta = base^{-\frac{2i}{head\_dim}}
        # (64,)
        inv_freq = 1.0 / (base ** (channel_range / head_dim))
        logger.debug(f"{inv_freq.shape=}")

        # 文字位置のインデックスを作成
        # (20480,)
        t = torch.arange(seq_len, dtype=torch.float32, device=device)
        logger.debug(f"{t.shape=}")
    
        # 回転角度を計算
        # 回転角度 = 文字位置 * 逆周波数
        # (20480, 64)
        freqs = torch.outer(t, inv_freq)
        logger.debug(f"{freqs.shape=}")

        # 回転角度からcosとsinを計算
        # (20480, 64), (20480, 64)
        cos, sin = freqs.cos(), freqs.sin()

        # bfloat16に変換
        cos, sin = cos.bfloat16(), sin.bfloat16()

        # ブロードキャストのために次元を追加
        # (1, 20480, 1, 64), (1, 20480, 1, 64)
        cos, sin = cos[None, :, None, :], sin[None, :, None, :]

        logger.debug(f"RoPEの事前計算完了 {cos.shape=} {sin.shape=}")
        return cos, sin

    def get_device(self):
        return self.transformer.wte.weight.device

    def estimate_flops(self):
        """
        モデルのトークンあたりのFLOPsを推定する
        FLOPSは1トークン生成するために必要な浮動小数点演算回数
        Chinchillaのスケール則で総訓練ステップ数を導くために必要
        https://arxiv.org/abs/2204.02311
        """
        logger.debug("モデルのFLOPsを推定開始")
    
        # モデルのパラメータ数を計算
        # 560,988,160 = 5.6億
        nparams = sum(p.numel() for p in self.parameters())
        logger.debug(f"{nparams=:,}")

        # 埋め込み層のパラメータ数を取得
        # 83,886,080 = 8,400万
        nparams_embedding = self.transformer.wte.weight.numel()
        logger.debug(f"{nparams_embedding=:,}")

        # レイヤー数l=20
        # ヘッド数h=10
        # クエリの次元q=128
        # シーケンス長t=2048
        l, h, q, t = self.config.n_layer, self.config.n_head, self.config.n_embd // self.config.n_head, self.config.sequence_len
        logger.debug(f"{l=}, {h=}, {q=}, {t=}")

        # FLOPsを計算
        # 3,491,758,080 = 3.5GFLOPs
        num_flops_per_token = 6 * (nparams - nparams_embedding) + 12 * l * h * q * t

        logger.debug(f"モデルのFLOPsの推定完了 {num_flops_per_token=:,}")
        return num_flops_per_token

    def setup_optimizers(self, unembedding_lr=0.004, embedding_lr=0.2, matrix_lr=0.02, weight_decay=0.0):
        """
        最適化関数を設定する

        Args:
            unembedding_lr (float): 出力層の学習率
            embedding_lr (float): 埋め込み層の学習率
            matrix_lr (float): トランスフォーマーの行列パラメータの学習率
            weight_decay (float): 重み減衰（L2正則化）係数
        Returns:
            List[torch.optim.Optimizer]: 設定された最適化関数のリスト
        """
        logger.debug(f"最適化関数を設定開始 {unembedding_lr=} {embedding_lr=} {matrix_lr=} {weight_decay=}")

        # 1) モデルの次元数を取得

        model_dim = self.config.n_embd

        ddp, rank, local_rank, world_size = get_dist_info()

        # トランスフォーマーの行列パラメータ
        matrix_params = list(self.transformer.h.parameters())
        logger.debug(f"{len(matrix_params)=}")

        # トランスフォーマーの埋め込みパラメータ
        embedding_params = list(self.transformer.wte.parameters())

        # 出力層のパラメータ
        lm_head_params = list(self.lm_head.parameters())

        assert len(list(self.parameters())) == len(matrix_params) + len(embedding_params) + len(lm_head_params)

        # 2) 入力の埋め込み層と出力の線形層に対してAdamWを初期化

        # モデルの次元に基づいて学習率をスケーリング
        dmodel_lr_scale = (model_dim / 768) ** -0.5
        if rank == 0:
            logger.debug(f"Scaling the LR for the AdamW parameters ∝1/√({model_dim}/768) = {dmodel_lr_scale:.6f}")

        adam_groups = [
            dict(params=lm_head_params, lr=unembedding_lr * dmodel_lr_scale),
            dict(params=embedding_params, lr=embedding_lr * dmodel_lr_scale),
        ]

        adamw_kwargs = dict(betas=(0.8, 0.95), eps=1e-10, weight_decay=weight_decay)

        AdamWFactory = DistAdamW if ddp else partial(torch.optim.AdamW, fused=True)

        adamw_optimizer = AdamWFactory(adam_groups, **adamw_kwargs)

        # 3) Transformerの行列パラメータに対してMuonを初期化

        muon_kwargs = dict(lr=matrix_lr, momentum=0.95)

        MuonFactory = DistMuon if ddp else Muon

        muon_optimizer = MuonFactory(matrix_params, **muon_kwargs)

        # 4) 2つの最適化関数を1つのリストにまとめる

        optimizers = [adamw_optimizer, muon_optimizer]

        for opt in optimizers:
            for group in opt.param_groups:
                group["initial_lr"] = group["lr"]

        logger.debug("最適化関数の設定完了")
        return optimizers

    def forward(self, idx, targets=None, kv_cache=None, loss_reduction='mean'):
        """
        GPTモデルの順伝搬

        Args:
            idx (Tensor): トークンIDのテンソル、形状は(batch_size, seq_len)
            targets (Tensor, optional): 目標トークンIDのテンソル、形状は(batch_size, seq_len)。デフォルトはNone。
            kv_cache (KVCache, optional): KVキャッシュオブジェクト。デフォルトはNone。
            loss_reduction (str, optional): 損失の集約方法。'mean'または'sum'。デフォルトは'mean'。
        Returns:
            Tensor:
                訓練モードの場合は損失テンソル（reductionに従う）
                推論モードの場合はロジットテンソル（batch_size, seq_len, vocab_size）
        """

        logger.debug(f"GPTモデルの順伝搬を実行 {idx.shape=} {idx.dtype=} {targets.shape if targets is not None else None=} {kv_cache if kv_cache is not None else None=}")

        B, T = idx.size()

        # 1) RoPEの準備

        assert T <= self.cos.size(1), f"Sequence length grew beyond the rotary embeddings cache: {T} > {self.cos.size(1)}"

        assert idx.device == self.cos.device, f"Rotary embeddings and idx are on different devices: {idx.device} != {self.cos.device}"

        assert self.cos.dtype == torch.bfloat16, "Rotary embeddings must be in bfloat16"

        # KVキャッシュが存在する場合、現在のキャッシュ内の位置にRoPEをオフセットする必要がある
        T0 = 0 if kv_cache is None else kv_cache.get_pos()

        # 事前計算されたRoPEを現在のシーケンス長に切り詰める
        # (1, 2048, 1, 64), (1, 2048, 1, 64)
        cos_sin = self.cos[:, T0:T0+T], self.sin[:, T0:T0+T]
        logger.debug(f"RoPEの切り出し完了 {cos_sin[0].shape=} {cos_sin[1].shape=}")

        # 2) 20層のTransformerの順伝播

        # トークンIDを埋め込みに変換
        x = self.transformer.wte(idx)

        # 最初のRMSNormを適用（Pre-LN）
        x = norm(x)

        # 各Transformerブロックを順番に適用
        for block in self.transformer.h:
            x = block(x, cos_sin, kv_cache)

        # 3) 出力の線形層を適用

        x = norm(x)

        # Logit Softcapping
        # ロジットの値が+/-15を超えないように制限する
        softcap = 15

        # 訓練モードの場合
        if targets is not None:

            # 出力の線形層を適用
            logits = self.lm_head(x)

            # logitsにソフトキャップを適用
            logits = softcap * torch.tanh(logits / softcap)

            # logitsをfloat32にアップキャスト
            logits = logits.float()

            # クロスエントロピー損失を計算
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),
                targets.view(-1),
                ignore_index=-1,
                reduction=loss_reduction
            )

            logger.debug(f"GPTモデルの順伝搬完了（訓練モード） {loss.shape=}")
            return loss

        # 推論モードの場合
        else:
            # 出力の線形層を適用
            logits = self.lm_head(x)

            # logitsにソフトキャップを適用
            logits = softcap * torch.tanh(logits / softcap)

            logger.debug(f"GPTモデルの順伝搬完了（推論モード） {logits.shape=}")
            return logits

    @torch.inference_mode() # 勾配計算を無効化
    def generate(self, tokens, max_tokens, temperature=1.0, top_k=None, seed=42):
        """
        テキストを生成するジェネレーター関数

        Args:
            tokens (List[int]): 初期トークンのリスト
            max_tokens (int): 生成する最大トークン数
            temperature (float, optional): 温度パラメータ。デフォルトは1.0。
            top_k (int, optional): Top-KサンプリングのK値。デフォルトはNoneで無効。
            seed (int, optional): 乱数シード。デフォルトは42。
        Yields:
            int: 生成された各トークンID
        """
        logger.debug(f"テキスト生成を開始 {tokens=} {max_tokens=} {temperature=} {top_k=} {seed=}")

        # 1) 初期化

        assert isinstance(tokens, list)

        device = self.get_device()
        logger.debug(f"{device=}")

        rng = None

        # 温度が設定されている場合、乱数生成器を初期化
        if temperature > 0:
            rng = torch.Generator(device=device)
            rng.manual_seed(seed)

        # バッチ次元を追加
        ids = torch.tensor([tokens], dtype=torch.long, device=device)

        # 2) トークン生成ループ

        # 最大トークン数まで生成
        for _ in range(max_tokens):

            # 1トークンを生成
            # (B, T, vocab_size)
            logits = self.forward(ids)

            # 直近のトークンのロジットを取得
            # (B, vocab_size)
            logits = logits[:, -1, :]

            # Top-Kサンプリングが有効な場合
            if top_k is not None:
                # 上位K個以外のロジットを無限小に設定
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')

            # 温度が設定されている場合
            if temperature > 0:
                # 確率分布をなめらかにしランダムに1つ選択
                logits = logits / temperature
                probs = F.softmax(logits, dim=-1)
                next_ids = torch.multinomial(probs, num_samples=1, generator=rng)

            # 温度が0の場合
            else:
                # 最も高い確率のトークンを選択（Greedy Decoding）
                next_ids = torch.argmax(logits, dim=-1, keepdim=True)

            # 生成したトークンをシーケンスの末尾に追加
            ids = torch.cat((ids, next_ids), dim=1)

            # 生成したトークンをCPUに移動してPythonのintに変換
            token = next_ids.item()

            logger.debug(f"生成トークン: {token}")
            yield token

### 推論エンジンの実装

In [ ]:
from collections import deque

In [ ]:
@torch.inference_mode()
def sample_next_token(logits, rng, temperature=1.0, top_k=None):
    """
    ロジットから次のトークンをサンプリングする
    Engineのgenerateメソッドで使用

    Args:
        logits (Tensor): ロジットのテンソル、形状は (B, vocab_size)
        rng (torch.Generator): 乱数生成器
        temperature (float, optional): 温度パラメータ。デフォルトは1.0。
        top_k (int, optional): Top-KサンプリングのK値。デフォルトはNoneで無効。
    Returns:
        Tensor: サンプリングされた次のトークンのテンソル、形状は (B, 1)
    """
    assert temperature >= 0.0, "temperature must be non-negative"

    # 1) 貪欲（Greedy）デコーディングの場合

    if temperature == 0.0:

        # 最も高い確率のトークンを選択
        return torch.argmax(logits, dim=-1, keepdim=True)

    # 2) サンプリングの場合

    if top_k is not None:

        # 上位KこのトークンIDとスコアを取得
        k = min(top_k, logits.size(-1))
        vals, idx = torch.topk(logits, k, dim=-1)

        # スコアに温度を適用
        vals = vals / temperature

        # 確率に変換
        probs = F.softmax(vals, dim=-1)

        # K個のトークンの中からランダムに1つ選択
        choice = torch.multinomial(probs, num_samples=1, generator=rng)

        # サンプリングされたインデックスを元のトークンIDに変換
        return idx.gather(1, choice)
    else:
        # 確率分布をなめらかにする
        logits = logits / temperature

        # 確率に変換
        probs = F.softmax(logits, dim=-1)

        # 全トークンからランダムに1つ選択
        return torch.multinomial(probs, num_samples=1, generator=rng)

In [ ]:
class KVCache:
    """
    推論を高速化するためのキー・バリューキャッシュ（KVキャッシュ）モジュール
    """

    def __init__(self, batch_size, num_heads, seq_len, head_dim, num_layers):
        logger.debug(f"KVCacheを初期化開始 {batch_size=} {num_heads=} {seq_len=} {head_dim=} {num_layers=}")

        # モデルの全層にKVキャッシュがある
        self.kv_shape = (num_layers, 2, batch_size, num_heads, seq_len, head_dim)

        self.kv_cache = None

        # キャッシュ内の書き込み位置
        self.pos = 0

        logger.debug("KVCacheの初期化完了")

    def reset(self):
        self.pos = 0

    def get_pos(self):
        return self.pos

    def prefill(self, other):
        """
        KVキャッシュを別のKVキャッシュで事前設定（prefill）する
        オプションでバッチ次元に沿って拡張することも可能
        複数のサンプルを並列に生成する場合に使用

        Args:
            other (KVCache): 事前設定に使用する別のKVキャッシュ
        """
        logger.debug(f"KVキャッシュを事前設定開始 {other.kv_cache is not None=}")

        # 1) 形状を検証

        assert self.kv_cache is None, "KVキャッシュが空でない場合、事前設定できません"
        assert other.kv_cache is not None, "事前設定に使用するKVキャッシュが空です"

        # 各次元を検証
        for ix, (dim1, dim2) in enumerate(zip(self.kv_shape, other.kv_shape)):
            # Transformerの層数、K/V、ヘッド数、ヘッド次元の検証
            if ix in [0, 1, 3, 5]:
                assert dim1 == dim2, f"Batch dim mismatch: {dim1} != {dim2}"

            # バッチサイズの検証
            elif ix == 2:
                # バッチサイズは拡張可能
                assert dim1 == dim2 or dim2 == 1, f"Batch dim mismatch: {dim1} != {dim2}"

            # シーケンス長の検証
            elif ix == 4:
                # シーケンス長は長い必要がある
                assert dim1 >= dim2, f"Seq len mismatch: {dim1} < {dim2}"

        # 2) キャッシュを初期化

        dtype, device = other.kv_cache.dtype, other.kv_cache.device
        self.kv_cache = torch.empty(self.kv_shape, dtype=dtype, device=device)

        # 3) データをコピー

        # バッチ次元に沿って複製
        self.kv_cache[:, :, :, :, :other.pos, :] = other.kv_cache

        # 4) 書き込み位置を更新

        self.pos = other.pos

        logger.debug("KVキャッシュの事前設定完了")

    def insert_kv(self, layer_idx, k, v):
        """
        KVキャッシュに新しいキーとバリューを挿入し、これまでの全てのキーとバリューを返す

        Args:
            layer_idx (int): 現在のTransformer層のインデックス
            k (Tensor): 新しいキーのテンソル、形状は (B, H, T_add, D)
            v (Tensor): 新しいバリューのテンソル、形状は (B, H, T_add, D)
        Returns:
            Tuple[Tensor, Tensor]: これまでの全てのキーとバリューのビュー、形状は (B, H, T_total, D)
        """
        logger.debug(f"KVキャッシュに挿入を実行 {layer_idx=} {k.shape=} {v.shape=} {self.pos=}")

        # 1) 初期化

        # KVキャッシュが初期化されていない場合
        if self.kv_cache is None:
            # KVキャッシュを初期化
            # データ型とデバイスが必要なため遅延初期化（lazy initialization）
            self.kv_cache = torch.empty(self.kv_shape, dtype=k.dtype, device=k.device)

        # Insert new keys/values to the cache and return the full cache so far

        # 現在の位置と追加するトークン数を取得
        B, H, T_add, D = k.size()

        # 書き込み開始位置t0と終了位置t1を計算 
        t0, t1 = self.pos, self.pos + T_add
        logger.debug(f"{t0=} {t1=}")

        # 書き込みの終了位置が初期化時のシーケンス長を超える場合、キャッシュを拡張
        if t1 > self.kv_cache.size(4):
            # 1024トークン分拡張
            t_needed = t1 + 1024

            # 1024の倍数に切り上げ
            t_needed = (t_needed + 1023) & ~1023

            additional_shape = list(self.kv_cache.shape)

            additional_shape[4] = t_needed - self.kv_cache.size(4)
            additional_cache = torch.empty(additional_shape, dtype=k.dtype, device=k.device)
            self.kv_cache = torch.cat([self.kv_cache, additional_cache], dim=4).contiguous()
            self.kv_shape = self.kv_cache.shape

        # 2) キーとバリューをキャッシュに挿入

        # キーをキャッシュに挿入
        self.kv_cache[layer_idx, 0, :, :, t0:t1] = k

        # バリューをキャッシュに挿入
        self.kv_cache[layer_idx, 1, :, :, t0:t1] = v

        # 3) これまでの全てのキーとバリューを返す

        # これまでの全てのキーを取得
        key_view = self.kv_cache[layer_idx, 0, :, :, :t1]

        # これまでの全てのバリューを取得
        value_view = self.kv_cache[layer_idx, 1, :, :, :t1]

        # 最後のTransformer層の場合、書き込み位置を更新
        if layer_idx == self.kv_cache.size(0) - 1:
            self.pos = t1

        logger.debug(f"KVキャッシュへの挿入完了 {key_view.shape=} {value_view.shape=} {self.pos=}")
        return key_view, value_view

In [ ]:
class RowState:
    """
    生成中の各行（サンプル）の状態を追跡するクラス
    """
    def __init__(self, current_tokens=None):

        # これまでに生成されたトークンIDの完全なリスト
        self.current_tokens = current_tokens or []

        # サンプリングしたトークンを無視し強制的に挿入するトークンのキュー
        # <|output_start|>などの特殊トークンの挿入に使用
        self.forced_tokens = deque()

        # <|python_start|>トークンを生成した時にTrueになるフラグ
        self.in_python_block = False

        # self.in_python_blockがTrueの間に生成されたトークンIDのリスト
        # Pythonのコードに対応するトークンID
        self.python_expr_tokens = []

        # 生成が完了したかどうかのフラグ
        # <|assistant_end|>や<|bos|>が生成された場合にTrueになる
        self.completed = False

In [ ]:
class Engine:
    """
    モデルの推論を実行するインターフェイス
    """

    def __init__(self, model, tokenizer):
        logger.debug(f"Engineを初期化開始 {model=} {tokenizer=}")
        
        self.model = model

        # ツール呼び出しで使用
        self.tokenizer = tokenizer

        logger.debug("Engineの初期化完了")

    @torch.inference_mode() # 勾配計算を無効化
    def generate(self, tokens, num_samples=1, max_tokens=None, temperature=1.0, top_k=None, seed=42):
        """
        トークンを1つずつストリーミングで生成するジェネレータ

        Args:
            tokens (List[int]): プロンプトのトークンIDのリスト
            num_samples (int): 生成するサンプル数（行数）
            max_tokens (int, optional): 生成する最大トークン数
            temperature (float): 温度パラメータ
            top_k (int, optional): Top-KサンプリングのK値
            seed (int): 乱数シード
        Yields:
            Tuple[List[int], List[int]]: 各行の次のトークンIDのリストと対応するマスクのリスト
        """
        logger.debug(f"ストリーミング生成を開始 {tokens=} {num_samples=} {max_tokens=} {temperature=} {top_k=} {seed=}")

        # 1) 初期化

        # 入力トークンの型を検証
        assert isinstance(tokens, list) and isinstance(tokens[0], int), "expecting list of ints"

        # デバイスを取得
        device = self.model.get_device()

        # 乱数生成器を初期化
        rng = torch.Generator(device=device)
        rng.manual_seed(seed)

        # 特殊トークンのIDを取得
        get_special = lambda s: self.tokenizer.encode_special(s)
        python_start = get_special("<|python_start|>")
        python_end = get_special("<|python_end|>")
        output_start = get_special("<|output_start|>")
        output_end = get_special("<|output_end|>")
        assistant_end = get_special("<|assistant_end|>") # if sampled, ends row
        bos = self.tokenizer.get_bos_token_id() # if sampled, ends row

        # 2) プロンプトのトークンを処理

        # KVキャッシュを初期化
        m = self.model.config
        kv_model_kwargs = {"num_heads": m.n_kv_head, "head_dim": m.n_embd // m.n_head, "num_layers": m.n_layer}
        kv_cache_prefill = KVCache(
            batch_size=1, # バッチサイズ1
            seq_len=len(tokens), # プロンプトの長さ
            **kv_model_kwargs, # KVキャッシュ設定
        )

        # トークンをテンソルに変換
        ids = torch.tensor([tokens], dtype=torch.long, device=device)
        logger.debug(f"{ids.shape=}")

        # モデルを順伝播してKVキャッシュを事前設定（prefill）
        logits = self.model.forward(ids, kv_cache=kv_cache_prefill)
        logger.debug(f"{logits.shape=}")

        # 最後のトークン（次の単語の予測）を抽出
        logits = logits[:, -1, :]
        logger.debug(f"{logits.shape=}")

        # 最初の1トークンをサンプリング    
        # (B, 1)
        next_ids = sample_next_token(logits, rng, temperature, top_k)
        logger.debug(f"{next_ids.shape=}")

        # サンプリングされたトークンをPythonのリストに変換
        sampled_tokens = next_ids[:, 0].tolist()
        logger.debug(f"{sampled_tokens=}")

        # 2) KVキャッシュの複製

        # 生成用のKVキャッシュを初期化
        kv_length_hint = (len(tokens) + max_tokens) if max_tokens is not None else self.model.config.sequence_len
        kv_cache_decode = KVCache(
            batch_size=num_samples,
            seq_len=kv_length_hint,
            **kv_model_kwargs,
        )
        logger.debug(f"{num_samples=} {kv_length_hint=}")

        # プロンプトで事前設定されたKVキャッシュを複製
        kv_cache_decode.prefill(kv_cache_prefill)

        # Prefill用のKVキャッシュは不要になったので削除
        del kv_cache_prefill

        # 3) 各サンプルの状態を初期化

        row_states = [RowState(tokens.copy()) for _ in range(num_samples)]
        logger.debug(f"{len(row_states)=}")

        # 4) 生成ループ

        # max_tokensに達するか、全サンプルが完了するまでトークンを生成
        num_generated = 0
        first_iteration = True
        while True:

            # 最大シーケンス長に達した場合、停止
            if max_tokens is not None and num_generated >= max_tokens:
                break

            # 全サンプルが完了した場合、停止
            if all(state.completed for state in row_states):
                break

            # 最初の生成ステップの場合
            if first_iteration:

                # プリフィルでサンプリングしたトークンを使用
                # すべてのサンプルにブロードキャスト
                sampled_tokens = [sampled_tokens[0]] * num_samples
                logger.debug(f"最初の生成ステップ {sampled_tokens=}")

                first_iteration = False

            # 2回目以降の生成ステップの場合
            else:

                # 順伝播を行い、各サンプルの次のトークンのロジットを計算
                # (B, T, vocab_size)
                logits = self.model.forward(ids, kv_cache=kv_cache_decode)
                logger.debug(f"2回目移行の生成ステップ {logits.shape=}")

                # 直近に生成されたトークンのロジットを抽出
                # (B, vocab_size)
                logits = logits[:, -1, :]
                logger.debug(f"{logits.shape=}")

                # 各サンプルの次のトークンをサンプリング
                # (B, 1)
                next_ids = sample_next_token(logits, rng, temperature, top_k)
                logger.debug(f"{next_ids.shape=}")

                # リストに変換
                sampled_tokens = next_ids[:, 0].tolist()
                logger.debug(f"{sampled_tokens=}")

            # それぞれのサンプルを処理
            token_column = [] # contains the next token id along each row
            token_masks = [] # contains the mask (was it sampled (1) or forced (0)?) along each row
            for i, state in enumerate(row_states):

                # 強制的に挿入するトークンがあるかを確認
                is_forced = len(state.forced_tokens) > 0

                # マスクを作成 
                # 0なら強制トークン、1ならサンプリングトークン
                token_masks.append(0 if is_forced else 1)

                # 強制トークンを優先して次のトークンを決定
                next_token = state.forced_tokens.popleft() if is_forced else sampled_tokens[i]

                # トークン列に次のトークンを追加
                token_column.append(next_token)

                # 状態の現在のトークン列に次のトークンを追加
                state.current_tokens.append(next_token)

                # 次のトークンが<|assistant_end|>または<|bos|>の場合
                if next_token == assistant_end or next_token == bos:
                    # 生成完了フラグを立てる
                    state.completed = True

                # <|python_start|>トークンが生成された場合
                if next_token == python_start:
                    # Pythonコードブロックに入るフラグを立てる
                    state.in_python_block = True
                    # Pythonコードトークンのリストを初期化
                    state.python_expr_tokens = []

                # <|python_end|>トークンが生成された場合
                elif next_token == python_end and state.in_python_block:
                    # Pythonコードブロックから出るフラグを下ろす
                    state.in_python_block = False

                    # ツール呼び出しを実行
                    if state.python_expr_tokens:
                        # トークン列をPythonコードの文字列にデコード
                        expr = self.tokenizer.decode(state.python_expr_tokens)

                        # ツールを呼び出す
                        result = use_calculator(expr)

                        # 結果がある場合
                        if result is not None:
                            # 結果をトークン化
                            result_tokens = self.tokenizer.encode(str(result))

                            # 特殊トークンで囲んで強制挿入キューに追加
                            # <|output_start|> result <|output_end|>
                            state.forced_tokens.append(output_start)
                            state.forced_tokens.extend(result_tokens)
                            state.forced_tokens.append(output_end)

                    # Pythonコードトークンのリストをクリア
                    state.python_expr_tokens = []

                # Pythonコードブロック内の場合
                elif state.in_python_block:
                    # 生成されたトークンをPythonコードトークンのリストに追加
                    state.python_expr_tokens.append(next_token)

            # 各行の次のトークン列とマスクをストリーミング出力
            yield token_column, token_masks

            # 生成トークン数を加算
            num_generated += 1

            # 次のループのためにトークンIDをテンソルに変換
            ids = torch.tensor(token_column, dtype=torch.long, device=device).unsqueeze(1)

    def generate_batch(self, tokens, num_samples=1, **kwargs):
        """
        非ストリーミング版

        Args:
            tokens (List[int]): プロンプトのトークンIDのリスト
            num_samples (int): 生成するサンプル数（行数）
            **kwargs: generateメソッドに渡す追加の引数
        Returns:
            Tuple[List[List[int]], List[List[int]]]: 各サンプルの生成されたトークン列と対応するマスクのリスト
        """
        logger.debug(f"バッチ生成を開始 {tokens=} {num_samples=} {kwargs=}")

        # 1) 初期化

        # ストップトークンのIDを取得
        assistant_end = self.tokenizer.encode_special("<|assistant_end|>")
        bos = self.tokenizer.get_bos_token_id()

        # 入力プロンプトを各サンプルにコピー
        results = [tokens.copy() for _ in range(num_samples)]

        # 各サンプルのマスクを初期化
        # 0は強制的に挿入されたトークン、1はサンプリングされたトークン
        masks = [[0] * len(tokens) for _ in range(num_samples)]

        # 各サンプルの生成完了フラグを初期化
        completed = [False] * num_samples

        # 2) 生成ループ

        # ストリーミングでトークンを生成
        for token_column, token_masks in self.generate(tokens, num_samples, **kwargs):

            # 各サンプルの生成されたトークンとマスクを収集
            for i, (token, mask) in enumerate(zip(token_column, token_masks)):

                # 未完了の場合
                if not completed[i]:

                    # 終了トークンの場合
                    if token == assistant_end or token == bos:

                        # 完了フラグを立てる
                        completed[i] = True

                    # 通常のトークンの場合
                    else:
                        # トークンを追加
                        results[i].append(token)

                        # マスクを追加
                        masks[i].append(mask)

            # すべての完了フラグが立っている場合、ループを終了
            if all(completed):
                break

        logger.debug(f"バッチ生成完了 {results=} {masks=}")
        return results, masks

### チェックポイントのロード

In [ ]:
import os
import re
import glob
import json
import logging
import torch

In [ ]:
def load_checkpoint(checkpoint_dir, step, device, load_optimizer=False):
    """
    指定されたステップのチェックポイントをロードする
    build_model関数で使用する

    Args:
        checkpoint_dir (str): チェックポイントディレクトリのパス
        step (int): ロードするステップ番号
        device (torch.device): モデルとオプティマイザの状態をロードするデバイス
        load_optimizer (bool, optional): オプティマイザの状態もロードするかどうか。デフォルトはFalse。
    Returns:
        Tuple[dict, dict or None, dict]: モデル状態、最適化関数の状態（存在する場合）、メタデータの辞書
    """

    # 1) モデルのチェックポイントをロード

    model_path = os.path.join(checkpoint_dir, f"model_{step:06d}.pt")
    model_data = torch.load(model_path, map_location=device)

    # 2) 最適化関数のチェックポイントをロード（必要な場合）

    optimizer_data = None

    if load_optimizer:
        optimizer_path = os.path.join(checkpoint_dir, f"optim_{step:06d}.pt")
        optimizer_data = torch.load(optimizer_path, map_location=device)

    # 3) メタデータをロード

    meta_path = os.path.join(checkpoint_dir, f"meta_{step:06d}.json")
    with open(meta_path, "r") as f:
        meta_data = json.load(f)

    return model_data, optimizer_data, meta_data

In [ ]:
def build_model(checkpoint_dir, step, device, phase):
    """
    チェックポイントからモデルを構築する
    load_model_from_dir関数で使用する

    Args:
        checkpoint_dir (str): チェックポイントディレクトリのパス
        step (int): ロードするステップ番号
        device (torch.device): モデルを配置するデバイス
        phase (str): モデルのフェーズ、"train"または"eval"
    Returns:
        Tuple[GPT, Tokenizer, dict]:
            - 未コンパイルでDDPでラップされていないベースモデル
            - トークナイザ
            - ベースモデルのトレーニング中に保存されたメタデータ
    """
    assert phase in ["train", "eval"], f"Invalid phase: {phase}"

    # 1) チェックポイントをロード

    model_data, optimizer_data, meta_data = load_checkpoint(checkpoint_dir, step, device, load_optimizer=False)

    model_data = {k.lstrip("_orig_mod."): v for k, v in model_data.items()}

    model_config_kwargs = meta_data["model_config"]

    logger.debug(f"Building model with config: {model_config_kwargs}")

    model_config = GPTConfig(**model_config_kwargs)

    # 2) モデルを初期化

    # メタデバイス上でモデルを初期化してメモリ使用量を削減
    with torch.device("meta"):
        model = GPT(model_config)

    # 3) モデルの状態をロード

    model.to_empty(device=device)

    # 重みを初期化（特にRoPEの初期化のため）
    model.init_weights()

    # チェックポイントからモデルの状態をロード
    model.load_state_dict(model_data, strict=True, assign=True)

    # モードを設定
    if phase == "eval":
        model.eval()
    else:
        model.train()

    # 4) トークナイザを初期化

    tokenizer = get_tokenizer()

    # 語彙サイズを検証
    assert tokenizer.get_vocab_size() == model_config_kwargs["vocab_size"]

    return model, tokenizer, meta_data

In [ ]:
def load_model_from_dir(checkpoints_dir, device, phase, model_tag=None, step=None):
    """
    ディレクトリからモデルをロードする
    load_modelで使用する

    Args:
        checkpoints_dir (str): チェックポイントディレクトリのパス
        device (torch.device): モデルを配置するデバイス
        phase (str): モデルのフェーズ、"train"または"eval"
        model_tag (str, optional): ロードするモデルのタグ。デフォルトはNoneで最大のモデルを推測。
        step (int, optional): ロードするステップ番号。デフォルトはNoneで最後のステップを推測。
    Returns:
        Tuple[GPT, Tokenizer, dict]:
            - 未コンパイルでDDPでラップされていないベースモデル
            - トークナイザ
            - ベースモデルのトレーニング中に保存されたメタデータ
    """

    # 1) モデルタグからモデルを特定する

    if model_tag is None:
        model_tag = find_largest_model(checkpoints_dir)
        logger.warn(f"No model tag provided, guessing model tag: {model_tag}")

    # 2) 最新のチェックポイントを特定する

    checkpoint_dir = os.path.join(checkpoints_dir, model_tag)
    if step is None:
        step = find_last_step(checkpoint_dir)

    assert step is not None, f"No checkpoints found in {checkpoint_dir}"
    logger.debug(f"Loading model from {checkpoint_dir} with step {step}")

    # 3) モデルを構築してロードする
    model, tokenizer, meta_data = build_model(checkpoint_dir, step, device, phase)

    return model, tokenizer, meta_data

In [ ]:
def load_model(source, *args, **kwargs):
    """
    事前定義されたソースからモデルをロードする
    頻繁に使用する

    Args:
        source (str): モデルのソース、"base"、"mid"、"sft"、"rl"のいずれか
        *args: load_model_from_dirに渡す追加の引数
        **kwargs: load_model_from_dirに渡す追加のキーワード引数

    Returns:
        Tuple[GPT, Tokenizer, dict]:
            - 未コンパイルでDDPでラップされていないベースモデル
            - トークナイザ
            - ベースモデルのトレーニング中に保存されたメタデータ
    """

    # 1) ソースに対応するモデルディレクトリを特定する

    model_dir = {
        "base": "base_checkpoints",
        "mid": "mid_checkpoints",
        "sft": "chatsft_checkpoints",
        "rl": "chatrl_checkpoints",
    }[source]

    base_dir = get_base_dir()

    checkpoints_dir = os.path.join(base_dir, model_dir)

    # 2)　ディレクトリからモデルをロードする

    return load_model_from_dir(checkpoints_dir, *args, **kwargs)

# 検証

device = torch.device("cuda")
model_tag = "d20"
step = 21400
model, tokenizer, meta = load_model("base", device, phase="train", model_tag=model_tag, step=step)

### 事前学習済みモデルの検証

In [ ]:
# 評価モードに変更
model.eval()

logger.setLevel(logging.INFO)

# サンプリング用のプロンプトを作成
prompts = [
    "Fukuoka city is",
    "The chemical symbol of gold is",
    "If yesterday was Friday, then tomorrow will be",
    "The opposite of hot is",
    "The planets of the solar system are:",
    "My favorite color is",
    "If 5*x + 3 = 13, then x is",
]

# 自動混合精度（Auto Mixed Precision, AMP）のコンテキストマネージャーを設定
autocast_ctx = torch.amp.autocast(device_type="cuda", dtype=torch.bfloat16)

# エンジンを初期化
engine = Engine(model, tokenizer)

# 各プロンプトに対してテキストを生成
for prompt in prompts:

    # トークン化
    tokens = tokenizer(prompt, prepend="<|bos|>")

    # 自動混合精度を有効化
    with autocast_ctx:

        # テキストを生成
        sample, _ = engine.generate_batch(
            tokens,
            num_samples=1,
            max_tokens=16, # 16トークン生成
            temperature=0 # 最も確率の高いトークンを選択
        )

    decoded = tokenizer.decode(sample[0])
    logger.info(f"ステップ数 {step=:05d} サンプル {decoded=}")

logger.setLevel(logging.DEBUG)

## 中間学習

中間学習はユーザーの発言を含む「全てのトークン」を使って、回答フォーマットや特殊トークンの扱いを訓練する

一般的な会話（SmolTalk）や多肢選択問題（MMLU）などの事前学習よりも高品質なデータを使用

In [ ]:
from collections import deque
from contextlib import nullcontext
import os
import time
import torch
import torch.distributed as dist
import wandb

In [ ]:
# 合成対話データをダウンロード

# 合成対話データはユーザーとアシスタントの対話データを含むJSONLファイル
# モデルに対話形式での応答を学習させるために使用

base_dir = get_base_dir()
identity_conversations_filepath = os.path.join(base_dir, "identity_conversations.jsonl")

IDENTITY_CONVERSATIONS_URL = "https://karpathy-public.s3.us-west-2.amazonaws.com/identity_conversations.jsonl"

if not os.path.exists(identity_conversations_filepath):
    !curl -L -o {identity_conversations_filepath} {IDENTITY_CONVERSATIONS_URL}

In [ ]:
# 対話合成データセットの確認

import json
identity_conversations = []
with open(identity_conversations_filepath, "r") as f:
    for line in f:
        identity_conversations.append(json.loads(line))

len(identity_conversations), identity_conversations[100]

# 996件

### ハイパーパラメータの設定

In [ ]:
# 並列実行するプロセス（GPU）の数
# デフォルトは8
NPROC_PER_NODE = 1

# WandBの実行名
# dummyの場合、WandBにログを送信しない
# デフォルトはdummy
run = "dummy"

# 計算に使用するデバイス（cuda, cpu, mps）
# 空文字列の場合、自動検出
device_type = ""

# ドライランモードの有効化フラグ
# 1の場合、WandBにログを送信するが、チェックポイントの保存やレポートは行わない
dry_run = 0

# ベースモデルの識別タグ
# d20など
model_tag = None

# 読み込むチェックポイントのステップ数
# 21400など
step = None

# 訓練の最大ステップ数
# デフォルトは-1（無制限）
num_iterations = 2

# モデルが処理できる最大シーケンス長
# デフォルトは2048トークン
max_seq_len = 2048

# 計算精度
dtype = "bfloat16"

# 1デバイスあたりのバッチサイズ
# デフォルトは32
device_batch_size = 1

# 論理的な総バッチサイズ
# 実際は勾配蓄積で分割して処理
total_batch_size = 524_288

# 最終層のAdamWの学習率
unembedding_lr = 0.004

# 入力埋め込み層のAdamWの学習率
embedding_lr = 0.2

# Transformer内部の行列のMuonの学習率
matrix_lr = 0.02

# 初期学習率の係数
init_lr_frac = 1.0

# 重み減衰率（L2正則化の強さ）
weight_decay = 0.0

# 評価のインターバルステップ数
# -1で無効化
eval_every = 150

# 評価時のトークン数
eval_tokens = 20 * 524_288

# ログ用
config_keys = [k for k,v in globals().items() if not k.startswith('_') and isinstance(v, (int, float, bool, str))]
user_config = {k: globals()[k] for k in config_keys}
user_config

### 分散処理の初期化

In [ ]:
def autodetect_device_type():
    """
    デバイスを自動検出する
    device_typeが指定されていない場合に使用する

    Returns:
        str: デバイスの種類（"cuda", "mps", "cpu"のいずれか）
    """
    if torch.cuda.is_available():
        device_type = "cuda"
    elif torch.backends.mps.is_available():
        device_type = "mps"
    else:
        device_type = "cpu"
    logger.debug(f"Autodetected device type: {device_type}")
    return device_type

autodetect_device_type()

In [ ]:
def get_dist_info():
    """
    分散データ並列（Distributed Data Parallel, DDP）環境の情報を取得する

    Returns:
        Tuple[bool, int, int, int]:
            - is_ddp (bool): DDP環境であるかどうか
            - ddp_rank (int): DDPのランク（プロセスID）
            - ddp_local_rank (int): DDPのローカルランク（ノード内のプロセスID）
            - ddp_world_size (int): DDPのワールドサイズ（総プロセス数）
    """
    if is_ddp():
        assert all(var in os.environ for var in ['RANK', 'LOCAL_RANK', 'WORLD_SIZE'])
        ddp_rank = int(os.environ['RANK'])
        ddp_local_rank = int(os.environ['LOCAL_RANK'])
        ddp_world_size = int(os.environ['WORLD_SIZE'])
        return True, ddp_rank, ddp_local_rank, ddp_world_size
    else:
        return False, 0, 0, 1

get_dist_info()

In [ ]:
def is_ddp():
    """
    分散データ並列（Distributed Data Parallel, DDP）環境であるかどうかを判定する
    """
    return int(os.environ.get('RANK', -1)) != -1

is_ddp()

In [ ]:
def compute_init(device_type="cuda"):
    """
    Pythonの実行環境を初期化し、分散データ並列（DDP）設定を行う
    並列環境を検証し、シード値と計算精度を設定する

    Args:
        device_type (str): 使用するデバイスの種類（"cuda", "mps", "cpu"のいずれか）
    Returns:
        Tuple[bool, int, int, int, torch.device]:
            - is_ddp (bool): DDP環境であるかどうか
            - ddp_rank (int): DDPのランク（プロセスID）
            - ddp_local_rank (int): DDPのローカルランク（ノード内のプロセスID）
            - ddp_world_size (int): DDPのワールドサイズ（総プロセス数）
            - device (torch.device): 使用するデバイス
    """

    # 1) 検証

    assert device_type in ["cuda", "mps", "cpu"], "Invalid device type atm"

    if device_type == "cuda":
        assert torch.cuda.is_available(), "Your PyTorch installation is not configured for CUDA but device_type is 'cuda'"
    if device_type == "mps":
        assert torch.backends.mps.is_available(), "Your PyTorch installation is not configured for MPS but device_type is 'mps'"

    # 2) 再現性の設定

    # シード値を固定
    torch.manual_seed(42)

    # CUDAのシード値を固定
    if device_type == "cuda":
        torch.cuda.manual_seed(42)

    # 決定的アルゴリズムの使用を有効化
    # 速度が低下する可能性があるためコメントアウト
    # torch.use_deterministic_algorithms(True)

    # 3) 計算精度の設定

    if device_type == "cuda":
        # bfloat16・float16の代わりにTF32を行列乗算に使用する
        torch.set_float32_matmul_precision("high")

    # 4) 分散データ並列（DDP）の設定

    ddp, ddp_rank, ddp_local_rank, ddp_world_size = get_dist_info()

    if ddp and device_type == "cuda":
        device = torch.device("cuda", ddp_local_rank)

        # CUDAをデフォルトデバイスに設定
        torch.cuda.set_device(device)

        # 分散通信ライブラリを初期化
        dist.init_process_group(backend="nccl", device_id=device)

        # 全プロセスがここまで到達するのを待機
        dist.barrier()
    else:
        # MPSまたはCPUの場合
        device = torch.device(device_type)

    if ddp_rank == 0:
        logger.debug(f"Distributed world size: {ddp_world_size}")

    return ddp, ddp_rank, ddp_local_rank, ddp_world_size, device

device_type = autodetect_device_type() if device_type == "" else device_type
ddp, ddp_rank, ddp_local_rank, ddp_world_size, device = compute_init(device_type)

In [ ]:
# 並列環境時のマスタープロセスの判定
master_process = ddp_rank == 0
logger.info(f"Master process: {master_process}")

In [ ]:
# 自動混合精度（Auto Mixed Precision, AMP）のコンテキストマネージャーを作成
# with autocast_ctx: でAMPを有効化できる

autocast_ctx = torch.amp.autocast(
    device_type=device_type,
    dtype=torch.bfloat16
) if device_type == "cuda" else nullcontext()

In [ ]:
# デバイスを同期する関数を作成
# synchronize() でデバイスを同期できる

synchronize = torch.cuda.synchronize \
    if device_type == "cuda" else lambda: None

In [ ]:
# 現在のVRAM使用量を取得する関数を作成

get_max_memory = torch.cuda.max_memory_allocated if device_type == "cuda" else lambda: 0

logger.info(f"最大メモリ使用量: {get_max_memory() / 1024**3:.2f} GB")

### WandBの初期化

In [ ]:
class DummyWandb:
    def __init__(self):
        pass
    def log(self, *args, **kwargs):
        pass
    def finish(self):
        pass

use_dummy_wandb = run == "dummy" or not master_process
logger.info(f"ダミーWandBの使用: {use_dummy_wandb}")

wandb_run = DummyWandb() \
    if use_dummy_wandb \
    else wandb.init(
        project="nanochat-mid",
        name=run,
        config=user_config)

### モデルの情報を抽出

In [ ]:
# メタデータから事前学習時のバッチサイズを取得

pretrain_batch_size = meta.get("device_batch_size", None)
logger.info(f"事前学習時のバッチサイズ: {pretrain_batch_size}")

# 事前学習時よりも大きいバッチサイズが指定された場合、警告を表示
if pretrain_batch_size is not None and device_batch_size > pretrain_batch_size:
    logger.debug(f"FOOTGUN WARNING: base model training used device_batch_size {pretrain_batch_size}, did you pass in a good --device_batch_size to this script?")

In [ ]:
# チェックポイントエクスポート用に元のモデルを保存
orig_model = model

# モデルをコンパイル
model = torch.compile(model, dynamic=False)

In [ ]:
# 統計情報を計算

depth = model.config.n_layer
logger.info(f"モデルの深さ: {depth}")

num_flops_per_token = model.estimate_flops()
logger.info(f"1トークンあたりの推論フロップ数: {num_flops_per_token / 1e9:.2f} GFLOPS")

tokens_per_fwdbwd = device_batch_size * max_seq_len
logger.info(f"デバイスあたりのトークン数（順伝播・逆伝播）: {tokens_per_fwdbwd:,}")

world_tokens_per_fwdbwd = tokens_per_fwdbwd * ddp_world_size
logger.info(f"ワールド全体のトークン数（順伝播・逆伝播）: {world_tokens_per_fwdbwd:,}")

# 論理的な総バッチサイズがワールド全体のトークン数で割り切れることを確認
assert total_batch_size % world_tokens_per_fwdbwd == 0

# 勾配蓄積ステップ数を計算
grad_accum_steps = total_batch_size // world_tokens_per_fwdbwd
logger.info(f"勾配蓄積ステップ数: {grad_accum_steps}")

In [ ]:
# BPE評価のためのトークンとバイト数のマッピングをロード

def get_token_bytes(device="cpu"):
    """
    トークンIDからバイト列へのマッピングをロードする

    Args:
        device (str or torch.device): マッピングをロードするデバイス
    Returns:
        Dict[int, bytes]: トークンIDからバイト列へのマッピング
    """

    # 1) トークンバイトマッピングファイルのパスを取得

    base_dir = get_base_dir()
    tokenizer_dir = os.path.join(base_dir, "tokenizer")
    token_bytes_path = os.path.join(tokenizer_dir, "token_bytes.pt")
    assert os.path.exists(token_bytes_path), f"Token bytes not found at {token_bytes_path}? It gets written by tok_train.py"

    # 2) トークンバイトマッピングをロード

    with open(token_bytes_path, "rb") as f:
        token_bytes = torch.load(f, map_location=device)

    return token_bytes

token_bytes = get_token_bytes(device=device)
len(token_bytes)

### 最適化関数

In [ ]:
import torch
from torch import Tensor
import torch.distributed as dist

In [ ]:
@torch.compile # PyTorch2.0のJITコンパイラを使用
def zeropower_via_newtonschulz5(G: Tensor, steps: int) -> Tensor:
    """
    ニュートン・シュルツ反復法で勾配の直行化を近似する

    行列Gの直交行列は、Gの特異値分解G = USV^Tに対してUV^Tで計算できる
    特異値分解（SVD）は計算コストが高いため近似手法を用いる
    原点における傾きを最大化するように選択された係数を持つ5次の反復法を採用

    Args: 
        G (Tensor): 直行化する行列、形状は(..., m, n)
        steps (int): 反復回数
    Returns:
        Tensor: 直交化された行列、形状は(..., m, n)
    """

    assert G.ndim >= 2

    # 5次反復法のための係数
    a, b, c = (3.4445, -4.7750,  2.0315)

    # 勾配をbfloat16にダウンキャスト
    X = G.bfloat16()

    # 安定化のため行列を横長にする
    if G.size(-2) > G.size(-1):
        X = X.mT

    # 行列のスペクトルノルム（最大の特異値）を1以下に正規化
    X = X / (X.norm(dim=(-2, -1), keepdim=True) + 1e-7)

    # ニュートン・シュワルツ反復法を適用
    for _ in range(steps):
        # A = X X^T
        A = X @ X.mT

        # B = b(X X^T) + c(X X^T)^2
        B = b * A + c * A @ A

        # X_k+1 = a X_k + (B) X_k 
        X = a * X + B @ X

    # 元の形状に戻す
    if G.size(-2) > G.size(-1):
        X = X.mT

    # 直交行列を返す
    return X

In [ ]:
class Muon(torch.optim.Optimizer):
    """
    Muon最適化関数 https://kellerjordan.github.io/posts/muon/

    内部的に標準的なSGDモーメンタムを実行し、その後に直交化の後処理ステップを実行する
    直行化ステップでは、各2Dパラメータの更新が最も近い直交行列に置き換えられる
    各更新を効率的に直交化するために、ニュートン・シュルツ反復を使用する
    これにより、GPU上でbfloat16で安定して実行できる利点がある

    注意:
    - この最適化関数は、埋め込み層、最終全結合層、0次元・1次元パラメータには使用しない（AdamWなどを使用）
    - 4Dの畳み込みフィルタに使用する場合、最後の3つの次元をフラット化すると機能する

    Args:
        lr (float): 内部SGDで使用される学習率
        momentum (float): 内部SGDで使用されるモーメンタム
        nesterov (bool): 内部SGDでNesterovスタイルのモーメンタムを使用するかどうか（推奨）
        ns_steps (int): ニュートン・シュルツ反復のステップ数
    """

    def __init__(self, params, lr=0.02, momentum=0.95, nesterov=True, ns_steps=5):
        """
        Muon最適化関数の初期化

        Args:
            params (iterable): 最適化するパラメータ
            lr (float, optional): 内部SGDで使用される学習率
            momentum (float, optional): 内部SGDで使用されるモーメンタム
            nesterov (bool, optional): 内部SGDでNesterovスタイルのモーメンタムを使用するかどうか
            ns_steps (int, optional): ニュートン・シュルツ反復のステップ数
        """
        # 最適化関数のデフォルト設定
        defaults = dict(lr=lr, momentum=momentum, nesterov=nesterov, ns_steps=ns_steps)

        params: list[Tensor] = [*params]

        param_groups = []

        # 同じ要素数のパラメータをグループ化
        for size in {p.numel() for p in params}:
            group = dict(params=[p for p in params if p.numel() == size])
            param_groups.append(group)

        # 親クラスの初期化
        super().__init__(param_groups, defaults)

    @torch.no_grad()
    def step(self):
        """
        Muon最適化関数の1ステップの更新を実行
        """

        # 各パラメータグループでループ
        for group in self.param_groups:

            params: list[Tensor] = group["params"]

            # 各パラメータでループ
            for p in params:
                # 勾配を取得
                g = p.grad
                assert g is not None

                # モメンタムバッファを取得または初期化
                state = self.state[p]
                if "momentum_buffer" not in state:
                    state["momentum_buffer"] = torch.zeros_like(g)
                buf: Tensor = state["momentum_buffer"]

                # モメンタムバッファを最新の勾配で更新（指数移動平均）
                buf.lerp_(g, 1 - group["momentum"])

                # ネステロフ・モーメンタムを使用する場合、勾配を調整
                g = g.lerp_(buf, group["momentum"]) if group["nesterov"] else buf

                # 勾配を直交化
                g = zeropower_via_newtonschulz5(g, steps=group["ns_steps"])

                # パラメータを更新
                # -group["lr"]: 学習率を負にして減少方向に更新
                # max(1, p.size(-2) / p.size(-1))**0.5: 行列の形状に基づいてスケーリング
                p.add_(g, alpha=-group["lr"] * max(1, p.size(-2) / p.size(-1))**0.5)

In [ ]:
# 最適化関数を初期化（線形層にはMuon、埋め込み層と最終全結合層にはAdamWを使用）
optimizers = model.setup_optimizers(
    unembedding_lr=unembedding_lr,
    embedding_lr=embedding_lr,
    matrix_lr=matrix_lr,
    weight_decay=weight_decay
)

# AdamWとMuonの最適化関数を取得
adamw_optimizer, muon_optimizer = optimizers

# 学習率と初期学習率を調整
for opt in optimizers:
    for group in opt.param_groups:
        # 学習率を初期学習率の係数でスケーリング
        group["lr"] = group["lr"] * init_lr_frac

        # 簡単のため初期学習率を保存
        group["initial_lr"] = group["lr"]

### タスク

In [ ]:
import random
from datasets import load_dataset

In [ ]:
def render_mc(question, letters, choices):
    """
    多肢選択式質問のプロンプトをレンダリングする

    Args:
        question (str): 質問文
        letters (List[str]): 選択肢のラベル（例: ["A", "B", "C", "D"]）
        choices (List[str]): 選択肢の内容（例: ["Paris", "London", "Berlin", "Madrid"]）

    Returns:
        str: レンダリングされた多肢選択式質問のプロンプト

    2つのテクニックを使用

    1. 「A) Paris」ではなく「Paris=A」の形式を使用
        - 小規模モデルの場合は、選択肢の後にラベルがある方が結びつきが良いため
    2. 「Paris = A」ではなく、「Paris=A」の形式を使用
        - " A"と"A"ではトークンIDが異なるため
    """
    query = f"Multiple Choice question: {question}\n"
    query += "".join([f"- {choice}={letter}\n" for letter, choice in zip(letters, choices)])
    query += "\nRespond only with the letter of the correct answer."
    return query

In [ ]:
class Task:
    """
    MMLU・GSM8Kなどのタスクのベースクラス
    データのスライスや評価用のメソッドのインターフェースを提供する
    """

    def __init__(self, start=0, stop=None, step=1):
        """
        Args:
            start (int):
                データセットの開始インデックス
            stop (int or None):
                データセットの終了インデックス
            step (int):
                データセットのステップサイズ
        """

        # 1) 引数の検証
        assert start >= 0, f"Start must be non-negative, got {start}"
        assert stop is None or stop >= start, f"Stop should be greater than or equal to start, got {stop} and {start}"
        assert step >= 1, f"Step must be strictly positive, got {step}"

        # 2) 属性の設定

        self.start = start
        self.stop = stop
        self.step = step

    @property
    def eval_type(self):
        """
        タスクの評価タイプ（generativeまたはcategorical）を返す
        """
        raise NotImplementedError

    def num_examples(self):
        raise NotImplementedError

    def get_example(self, index):
        raise NotImplementedError

    def __len__(self):
        """
        見かけ上のデータ数を返す
        """
        start = self.start
        stop = self.num_examples() if self.stop is None else self.stop
        step = self.step
        span = stop - start
        num = (span + step - 1) // step # 切り上げ除算
        assert num >= 0, f"Negative number of examples???: {num}"
        return num

    def __getitem__(self, index: int):
        """
        インデックスを指定してデータを取得する
        """
        assert isinstance(index, int), f"Index must be an integer, got {type(index)}"
        physical_index = self.start + index * self.step
        conversation = self.get_example(physical_index)
        return conversation

    def evaluate(self, problem, completion):
        """
        タスクの評価を行う
        """
        raise NotImplementedError

#### SmolTalk

In [ ]:
class SmolTalk(Task):
    """
    SmolTalkデータセットを訓練で使用するためのクラス
    SmolTalkはHuggingFaceが提供する対話データセットで一般的な対話データセット
    46万件の訓練データと2.4万件のテストデータを含む
    Smolバージョンはモデルサイズが小さい場合に適している
    中身はcontentとroleフィールドを持つ辞書のリスト
    チャットボットとしての基本的な振る舞いを学習するのに適している

    SmolTalk:
    https://huggingface.co/datasets/HuggingFaceTB/smol-smoltalk
    """

    def __init__(self, split, **kwargs):
        super().__init__(**kwargs)
        assert split in ["train", "test"], "SmolTalk split must be train|test"

        # データセットをロードしてシャッフル
        self.ds = load_dataset(
            "HuggingFaceTB/smol-smoltalk", split=split
        ).shuffle(seed=42)

        self.length = len(self.ds)

    def num_examples(self):
        return self.length

    def get_example(self, index):
        """
        インデックスを指定してSmolTalkの対話データを取得する
        """

        # 1) データの取得

        # データセットから行を取得
        row = self.ds[index]

        # メッセージ列を取得
        messages = row["messages"]

        # 2) データの検証

        assert len(messages) >= 1

        first_message = messages[0]

        # 最初のメッセージがシステムメッセージの場合
        if first_message["role"] == "system":
            # スキップ
            rest_messages = messages[1:]
        else:
            rest_messages = messages

        # 少なくとも2つのメッセージであるかを検証
        assert len(rest_messages) >= 2, "SmolTalk messages must have at least 2 messages"

        # メッセージごとに検証
        for i, message in enumerate(rest_messages):

            # ユーザー -> アシスタント -> ユーザー -> ... の順序であることを検証
            expected_role = "user" if i % 2 == 0 else "assistant"
            assert message["role"] == expected_role, \
                f"Message {i} has role {message['role']} but should be {expected_role}"

            # contentフィールドが文字列であることを検証
            assert isinstance(message["content"], str), \
                "Content must be a string"

        # 3) 整形

        conversation = {
            "messages": messages,
        }

        return conversation


smoltalk = SmolTalk(split="train", start=0, stop=5)
first_conversation = smoltalk[0]
first_conversation

#### MMLU

In [ ]:
class MMLU(Task):
    """
    MMLU（Massive Multitask Language Understanding）データセットを扱うクラス
    MMLUは57の学術的な科目にわたる多数の選択式質問を含むデータセット
    各質問には4つの選択肢があり、正しい答えはA、B、C、Dのいずれか
    MMLUは言語モデルの幅広い知識と推論能力を評価するために使用される
    allサブセットは11万6千件、auxiliary_trainサブセットは1万件の追加訓練データを含む
    各行はquestion, subject, choices, answerフィールドを持つ
    データセットはHuggingFaceの"cais/mmlu"からロードされる
    https://huggingface.co/datasets/cais/mmlu
    """

    # 定数: 選択肢の文字
    letters = ('A', 'B', 'C', 'D')

    # 定数: MMLUの57科目のグループ
    groups = ('abstract_algebra', 'anatomy', 'astronomy', 'business_ethics', 'clinical_knowledge', 'college_biology', 'college_chemistry', 'college_computer_science', 'college_mathematics', 'college_medicine', 'college_physics', 'computer_security', 'conceptual_physics', 'econometrics', 'electrical_engineering', 'elementary_mathematics', 'formal_logic', 'global_facts', 'high_school_biology', 'high_school_chemistry', 'high_school_computer_science', 'high_school_european_history', 'high_school_geography', 'high_school_government_and_politics', 'high_school_macroeconomics', 'high_school_mathematics', 'high_school_microeconomics', 'high_school_physics', 'high_school_psychology', 'high_school_statistics', 'high_school_us_history', 'high_school_world_history', 'human_aging', 'human_sexuality', 'international_law', 'jurisprudence', 'logical_fallacies', 'machine_learning', 'management', 'marketing', 'medical_genetics', 'miscellaneous', 'moral_disputes', 'moral_scenarios', 'nutrition', 'philosophy', 'prehistory', 'professional_accounting', 'professional_law', 'professional_medicine', 'professional_psychology', 'public_relations', 'security_studies', 'sociology', 'us_foreign_policy', 'virology', 'world_religions')

    def __init__(self, subset, split, **kwargs):
        super().__init__(**kwargs)

        # 1) 引数の検証

        # サブセットはallまたはauxiliary_trainのみ指定可能
        # auxiliary_trainは追加の訓練データ
        assert subset in ["all", "auxiliary_train"], f"subset {subset} must be all|auxiliary_train"

        # スプリットはtrain、validation、dev、testのいずれか
        assert split in ["train", "validation", "dev", "test"], f"split {split} must be train|validation|dev|test"

        # auxiliary_trainサブセットはtrainスプリットのみ許可
        if subset == "auxiliary_train":
            assert split == "train", "auxiliary_train must be split into train"

        # 2) 属性の設定

        self.subset = subset
        self.split = split

        # 3) データセットのロード

        self.ds = load_dataset("cais/mmlu", subset, split=split).shuffle(seed=42)

        # 4) データセットの前処理

        if subset == "auxiliary_train":
            # なぜかauxiliary_trainの行には奇妙な追加の'train'ラッパーがあるため削除
            self.ds = self.ds.map(lambda row: row['train'], remove_columns=['train'])

    @property
    def eval_type(self):
        return 'categorical'

    def num_examples(self):
        return len(self.ds)

    def get_example(self, index):
        """
        インデックスを指定してMMLUの問題データを取得する
        """

        # 1) データの取得

        # データセットから行を取得
        row = self.ds[index]

        # 列を取得
        question = row["question"]
        choices = row["choices"]
        answer = row["answer"]
        subject = row["subject"]

        # 選択肢は4つであることを検証
        assert len(choices) == 4, "MMLU should have 4 choices"

        # 質問文と選択肢を対話オブジェクトに変換
        user_message = render_mc(question, self.letters, choices)

        # 正しい答えの選択肢の文字を取得
        # 0 -> A, ...
        assistant_message = self.letters[answer]

        # 2) 対話形式に整形

        messages = [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": assistant_message}
        ]

        conversation = {
            "messages": messages,
            "subject": subject, # 評価時に使用
            "letters": self.letters, # 評価時に使用
        }

        return conversation

    def evaluate(self, conversation, assistant_response):
        """
        モデルが生成した回答を評価

        Args:
            conversation (dict):
                get_exampleから取得した対話データ
            assistant_response (str):
                モデルが生成した回答（例: "A"）
        Returns:
            bool:
                モデルの回答が正しい場合にTrue、そうでない場合にFalse
        """
        # 1) 回答の検証

        assert assistant_response in self.letters, f"MMLU answer {assistant_response} is expected to be one of {self.letters}"

        # 2) 正解の取得

        # Aなどの正しい答えの文字を取得
        assistant_message = conversation['messages'][-1]['content']

        # 3) 正誤の判定
        return assistant_response == assistant_message

mmlu = MMLU(subset="all", split="test", start=0, stop=5)
first_mmlu_example = mmlu[0]
first_mmlu_example, mmlu.evaluate(first_mmlu_example, "A")

#### GSM

In [ ]:
GSM_RE = re.compile(r"#### (\-?[0-9\.\,]+)")

In [ ]:
def extract_answer(completion):
    """
    対話から数値の答えを抽出する
    答えは「####」マーカーの後に続く数値

    # 公式のコードはこちら
    https://github.com/openai/grade-school-math/blob/3101c7d5072418e28b9008a6636bde82a006892c/grade_school_math/dataset.py#L28
    """
    match = GSM_RE.search(completion)
    if match:
        match_str = match.group(1).strip()
        match_str = match_str.replace(",", "")
        return match_str
    return None

In [ ]:
class GSM8K(Task):
    """
    OpenAIのGSM8K（Grade School Math 8K）データセットを扱うクラス
    GSM8Kは8,700問の小学校レベルの数学問題を含むデータセット
    回答には計算過程があり、それをPythonツール呼び出しとして再構成する
    https://huggingface.co/datasets/openai/gsm8k
    """

    def __init__(self, subset, split, **kwargs):
        super().__init__(**kwargs)

        # 1) 検証

        assert subset in ["main", "socratic"], "GSM8K subset must be main|socratic"

        assert split in ["train", "test"], "GSM8K split must be train|test"

        # 2) データセットのロード

        self.ds = load_dataset("openai/gsm8k", subset, split=split).shuffle(seed=42)

    @property
    def eval_type(self):
        return 'generative'

    def num_examples(self):
        return len(self.ds)

    def get_example(self, index):
        """
        インデックスを指定して問題を取得する
        """

        # 1) データの取得

        row = self.ds[index]

        question = row['question']
        answer = row['answer']

        # 2) 対話形式に整形

        # アシスタントメッセージのパーツを格納するリスト
        assistant_message_parts = []

        # 計算部分を分解
        # ツール呼び出しは <<expression=result>> の形式で埋め込まれている
        # 例: "The answer is <<2+2=4>>."
        # -> ["The answer is ", "<<2+2=4>>", "."]
        parts = re.split(r'(<<[^>]+>>)', answer)

        # 各パーツを処理
        for part in parts:

            # ツール呼び出しのパーツの場合
            if part.startswith('<<') and part.endswith('>>'):
                # <<と>>を取り除く
                inner = part[2:-2]

                # =が含まれている場合、式と結果に分割
                if '=' in inner:
                    expr, result = inner.rsplit('=', 1)
                else:
                    expr, result = inner, ""

                # ツール呼び出しをアシスタントメッセージのパーツとして追加
                assistant_message_parts.append({"type": "python", "text": expr})

                # ツール呼び出しの結果をアシスタントメッセージのパーツとして追加
                assistant_message_parts.append({"type": "python_output", "text": result})

            # 通常のテキストパーツの場合
            else:
                # アシスタントメッセージのパーツとして追加
                assistant_message_parts.append({"type": "text", "text": part})

        # ユーザーメッセージとアシスタントメッセージをまとめる
        # アシスタントメッセージはパーツのリストとして格納
        messages = [
            {"role": "user", "content": question},
            {"role": "assistant", "content": assistant_message_parts},
        ]

        conversation = {
            "messages": messages,
        }

        return conversation

    def evaluate(self, conversation, assistant_response):
        """
        対話とモデルの応答から評価結果を返す

        Args:
            conversation (dict):
                get_exampleから取得した対話データ
            assistant_response (str):   
                モデルが生成した応答
        Returns:
            int:
                評価結果（0 = 不正解、1 = 正解）
        """

        # 1) 応答の検証

        assert isinstance(assistant_response, str), "Assuming simple string response for now"

        # 正しい答えが含まれる最後のアシスタントメッセージを取得
        assistant_message = conversation['messages'][-1]

        assert assistant_message['role'] == "assistant", "Last message must be from the Assistant"

        # パーツのリストであることを検証
        assert isinstance(assistant_message['content'], list), "This is expected to be a list of parts"

        # 2) 正解と予測の抽出

        # 最後のアシスタントメッセージのテキストパーツを取得
        last_text_part = assistant_message['content'][-1]['text']

        # 正解の数値を抽出
        ref_num = extract_answer(last_text_part)

        # 予測の数値を抽出
        pred_num = extract_answer(assistant_response)

        # 3) 正誤の判定

        is_correct = int(pred_num == ref_num)
        return is_correct

    def reward(self, conversation, assistant_response):
        """
        強化学習で使用する報酬を計算する
        報酬は0.0または1.0の浮動小数点数で表される
        """
        is_correct = self.evaluate(conversation, assistant_response)
        is_correct_float = float(is_correct)
        return is_correct_float

gsm = GSM8K(subset="main", split="test", start=0, stop=5)
first_gsm_example = gsm[0]
first_gsm_example, gsm.evaluate(first_gsm_example, "The answer is 109. #### 109")


#### CustomJSON

In [ ]:
class CustomJSON(Task):
    """
    JSONLファイルから対話データをロードするクラス
    各行はroleとcontentフィールドを持つメッセージオブジェクトのJSON配列である必要がある
    例: [{"role":"user","content":"Hi"},{"role":"assistant","content":"Hello"}]
    """

    def __init__(self, filepath, **kwargs):
        super().__init__(**kwargs)

        # 1) 属性の設定

        self.filepath = filepath
        self.conversations = []

        # 2) 全ての対話データをJSONLファイルからロード

        # ファイルが存在しない場合
        if not os.path.exists(filepath):
            print("-" * 80)
            print(f"Warning: File {filepath} does not exist")
            print("HINT (Oct 21 2025)")
            print("If you recently did a git pull and suddely see this, it might be due to the new addition of identity conversations")
            print("See this discussion for more details: https://github.com/karpathy/nanochat/discussions/139")
            print("Quick fix: simply run the following command to download the file and you're done:")
            print(f"curl -L -o {filepath} https://karpathy-public.s3.us-west-2.amazonaws.com/identity_conversations.jsonl")
            print("-" * 80)


        else:

            # ファイルを開いて各行を処理
            with open(filepath, 'r') as f:

                for line in f:
                    line = line.strip()

                    # 空行の場合はスキップ
                    if not line:
                        continue

                    # JSONをパース
                    messages = json.loads(line)

                    # 対話データの検証

                    assert isinstance(messages, list), f"Expected list of messages, got {type(messages)}"

                    assert len(messages) >= 2, f"Conversation must have at least 2 messages, got {len(messages)}"

                    for i, message in enumerate(messages):
                        assert "role" in message, f"Message {i} missing 'role' field"

                        assert "content" in message, f"Message {i} missing 'content' field"

                        expected_role = "user" if i % 2 == 0 else "assistant"

                        assert message["role"] == expected_role, f"Message {i} has role {message['role']} but should be {expected_role}"

                        assert isinstance(message["content"], str), f"Message {i} content must be a string"


                    # 対話データをリストに追加
                    self.conversations.append(messages)

        self.length = len(self.conversations)

    def num_examples(self):
        return self.length

    def get_example(self, index):
        """
        インデックスを指定して対話データを取得する
        """

        messages = self.conversations[index]

        conversation = {
            "messages": messages,
        }

        return conversation

custom_data = CustomJSON(filepath=identity_conversations_filepath, start=0, stop=5)
first_custom_example = custom_data[0]
first_custom_example

#### SimpleSpelling

In [ ]:
WORD_LIST_URL = "https://raw.githubusercontent.com/dwyl/english-words/refs/heads/master/words_alpha.txt"

class SimpleSpelling(Task):
    """
    単語のスペルを練習させるための非常にシンプルなタスク
    トークナイザーでトークン化された単語を個別の文字として認識できるようにするため
    """

    def __init__(self, size=1000, split="train", **kwargs):
        super().__init__(**kwargs)

        # 1) 引数の検証

        assert split in ["train", "test"], "SpellingBee split must be train|test"

        # 2) 属性の設定

        self.size = size
        self.split = split

        # 3) 単語リストのロード

        # 単語リストのファイル名を取得
        filename = WORD_LIST_URL.split("/")[-1]

        # 単語リストをダウンロード
        word_list_path = download_file_with_lock(WORD_LIST_URL, filename)

        # 単語リストを読み込み
        with open(word_list_path) as f:
            words = [line.strip() for line in f]

        # 4) 単語リストをシャッフル

        rng = random.Random(42)
        rng.shuffle(words)

        # 5) 単語を属性に保存

        self.words = words

    @property
    def eval_type(self):
        return 'generative'

    def num_examples(self):
        return self.size

    def get_example(self, index):
        """
        インデックスを指定してスペル練習の問題を取得する
        """

        # 1) 乱数シードの設定

        # インデックスをシードとして扱い、同じインデックスで同じ単語が得られるようにする
        seed = index if self.split == "train" else -(index + 1)

        rng = random.Random(seed)

        # 2) ランダムな単語の選択

        word = rng.choice(self.words)

        # 3) 対話形式に整形

        # 単語を文字ごとに分割してカンマで結合
        word_letters = ",".join(list(word))

        # Spell the word: apple -> apple:a,p,p,l,e のような形式で対話を作成
        messages = [
            {"role": "user", "content": f"Spell the word: {word}"},
            {"role": "assistant", "content": f"{word}:{word_letters}"}
        ]

        conversation = {
            "messages": messages,
        }

        return conversation

simple_spelling = SimpleSpelling(size=5, split="test", start=0, stop=5)
first_spelling_example = simple_spelling[0]
first_spelling_example

#### SpellingBee

In [ ]:
# アルファベットの文字
LETTERS = "abcdefghijklmnopqrstuvwxyz"

# 37万語の多様な英単語リスト
WORD_LIST_URL = "https://raw.githubusercontent.com/dwyl/english-words/refs/heads/master/words_alpha.txt"

# GSM8Kと同じ回答抽出正規表現
ANSWER_RE = re.compile(r"#### (\-?[0-9\.\,]+)")

def extract_answer(completion):
    """
    回答を抽出する
    #### の後の数値が回答
    """
    match = ANSWER_RE.search(completion)
    if match:
        match_str = match.group(1).strip()
        match_str = match_str.replace(",", "")
        return match_str
    return None

# データ拡張用のユーザーメッセージテンプレート
USER_MSG_TEMPLATES = [
    "How many {letter} are in the word {word}",
    "How many {letter} are in {word}",
    "Count the number of {letter} in {word}",
    "How many times does {letter} appear in {word}",
    "What's the count of {letter} in {word}",
    "In the word {word}, how many {letter} are there",
    "How many letter {letter} are in the word {word}",
    "Count how many {letter} appear in {word}",
    "Tell me the number of {letter} in {word}",
    "How many occurrences of {letter} are in {word}",
    "Find the count of {letter} in {word}",
    "Can you count the {letter} letters in {word}",
    "What is the frequency of {letter} in {word}",
    "How many {letter}s are in {word}",
    "How many {letter}'s are in {word}",
    "Count all the {letter} in {word}",
    "How many times is {letter} in {word}",
    "Number of {letter} in {word}",
    "Total count of {letter} in {word}",
    "How many {letter} does {word} have",
    "How many {letter} does {word} contain",
    "What's the number of {letter} in {word}",
    "{word} has how many {letter}",
    "In {word}, count the {letter}",
    "How many {letter} appear in {word}",
    "Count the {letter} in {word}",
    "Give me the count of {letter} in {word}",
    "How many instances of {letter} in {word}",
    "Show me how many {letter} are in {word}",
    "Calculate the number of {letter} in {word}",
    # Spanish
    "¿Cuántas {letter} hay en {word}?",
    "¿Cuántas veces aparece {letter} en {word}?",
    "Cuenta las {letter} en {word}",
    "¿Cuántas letras {letter} tiene {word}?",
    # Chinese (Simplified)
    "{word}中有多少个{letter}",
    "{word}里有几个{letter}",
    "数一下{word}中的{letter}",
    "{word}这个词里有多少{letter}",
    # Korean
    "{word}에 {letter}가 몇 개 있나요",
    "{word}에서 {letter}의 개수는",
    "{word}에 {letter}가 몇 번 나오나요",
    "{word}라는 단어에 {letter}가 몇 개",
    # French
    "Combien de {letter} dans {word}",
    "Combien de fois {letter} apparaît dans {word}",
    "Compte les {letter} dans {word}",
    # German
    "Wie viele {letter} sind in {word}",
    "Wie oft kommt {letter} in {word} vor",
    "Zähle die {letter} in {word}",
    # Japanese
    "{word}に{letter}は何個ありますか",
    "{word}の中に{letter}がいくつ",
    "{word}に{letter}が何回出てくる",
]

class SpellingBee(Task):
    """
    文字を数えるタスクを扱うクラス
    与えられた単語の中に特定の文字がいくつ含まれているかを数える
    例えば、単語 "apple" に対して文字 "p" が2つ含まれている
    """

    def __init__(self, size=1000, split="train", **kwargs):
        super().__init__(**kwargs)

        # 1) 引数の検証

        assert split in ["train", "test"], "SpellingBee split must be train|test"

        # 2) 属性の設定

        self.size = size
        self.split = split

        # 3) 単語リストのロード

        filename = WORD_LIST_URL.split("/")[-1]

        word_list_path = download_file_with_lock(WORD_LIST_URL, filename)

        with open(word_list_path) as f:
            words = [line.strip() for line in f]

        # 属性に単語を保存
        self.words = words

    @property
    def eval_type(self):
        return 'generative'

    def num_examples(self):
        return self.size

    def get_example(self, index):
        """
        インデックスを指定して問題を取得する
        """

        # インデックスをシードとして使用
        seed = index if self.split == "train" else -(index + 1)
        rng = random.Random(seed)

        # ランダムに単語を選択
        word = rng.choice(self.words)

        # 文字は90%の確率で単語から選ばれ、10%の確率でランダムに選ばれる
        letter = rng.choice(word) if rng.random() < 0.9 else rng.choice(LETTERS)

        # 文字数を数える
        count = word.count(letter)

        # テンプレートからランダムに選択
        template = rng.choice(USER_MSG_TEMPLATES)

        # 30%の確率でテンプレートを小文字にする
        if rng.random() < 0.3:
            template = template.lower()

        # 引用符のオプション
        quote_options = ['', "'", '"']

        # 文字の引用符をランダムに選択
        letter_quote = rng.choice(quote_options)

        # 単語の引用符をランダムに選択
        word_quote = rng.choice(quote_options)

        # ユーザーメッセージ用の文字を引用符で囲む
        letter_wrapped = f"{letter_quote}{letter}{letter_quote}"

        # 単語を引用符で囲む
        word_wrapped = f"{word_quote}{word}{word_quote}"

        # ユーザーメッセージをテンプレートに埋め込む
        user_msg = template.format(letter=letter_wrapped, word=word_wrapped)

        # 50%の確率でユーザーメッセージに疑問符を追加
        if rng.random() < 0.5:
            user_msg += "?"

        # アシスタントの正解の回答を作成 - パーツ（テキスト + ツール呼び出し）として構築
        assistant_parts = []

        # 単語の文字列をカンマで区切って作成
        word_letters = ",".join(list(word))

        # 手動でカウントするパーツ
        manual_text = f"""We are asked to find the number '{letter}' in the word '{word}'. Let me try a manual approach first.

First spell the word out:
{word}:{word_letters}

Then count the occurrences of '{letter}':
"""

        # 手動でカウントするループ
        running_count = 0
        for i, char in enumerate(word, 1):
            # 対象の文字と一致する場合
            if char == letter:
                running_count += 1
                manual_text += f"{i}:{char} hit! count={running_count}\n"
            else:
                manual_text += f"{i}:{char}\n"

        # 手動でのカウント結果のパーツ
        manual_text += f"\nThis gives us {running_count}."

        # アシスタントメッセージに手動でのカウントを追加
        assistant_parts.append({"type": "text", "text": manual_text})

        # Pythonツール呼び出しでダブルチェックするパーツ
        assistant_parts.append({"type": "text", "text": "\n\nLet me double check this using Python:\n\n"})

        # Pythonの式を作成
        python_expr = f"'{word}'.count('{letter}')"

        # Pythonツール呼び出しのパーツを追加
        assistant_parts.append({"type": "python", "text": python_expr})

        # Pythonの出力パーツを追加
        assistant_parts.append({"type": "python_output", "text": str(count)})

        # 最終的な回答パーツを追加
        assistant_parts.append({"type": "text", "text": f"\n\nPython gives us {count}.\n\nMy final answer is:\n\n#### {count}"})

        # 全ての結果をまとめて対話形式に整形

        messages = [
            {"role": "user", "content": user_msg},
            {"role": "assistant", "content": assistant_parts}
        ]

        conversation = {
            "messages": messages,
        }

        return conversation

    def evaluate(self, conversation, assistant_response):
        """
        モデルの回答が正しいかどうかを評価する
        GSM8Kと同様の評価方法を使用
        """

        # 1) 応答の検証

        assert isinstance(assistant_response, str), "Assuming simple string response for now"

        # 2) 対話から正しい答えを抽出

        assistant_message = conversation['messages'][-1]

        assert assistant_message['role'] == "assistant", "Last message must be from the Assistant"

        assert isinstance(assistant_message['content'], list), "This is expected to be a list of parts"

        last_text_part = assistant_message['content'][-1]['text']
        ref_num = extract_answer(last_text_part)

        # 3) モデルの予測から答えを抽出

        pred_num = extract_answer(assistant_response)

        # 4) 正誤の判定

        # intにキャストして返す
        is_correct = int(pred_num == ref_num)
        return is_correct

    def reward(self, conversation, assistant_response):
        """
        強化学習で使用する報酬を計算する
        """
        is_correct = self.evaluate(conversation, assistant_response)

        # 報酬を浮動小数点数で返す
        is_correct_float = float(is_correct)
        return is_correct_float


spelling_bee = SpellingBee(size=5, split="test", start=0, stop=5)
first_spelling_bee_example = spelling_bee[0]
first_spelling_bee_example, spelling_bee.evaluate(first_spelling_bee_example, "#### 2")

In [ ]:
class TaskMixture(Task):
    """
    複数のタスクを混ぜ合わせて1つのタスクとして扱うクラス
    SFTで使用
    """

    def __init__(self, tasks, **kwargs):
        super().__init__(**kwargs)

        # タスクのリストを属性に保存
        self.tasks = tasks

        # 各タスクの会話数を計算
        self.lengths = [len(task) for task in self.tasks]

        # 全タスクの会話数の合計を計算
        self.num_conversations = sum(self.lengths)

        # タスクのインデックスとデータのインデックスのマップを作成
        self.index_map = []
        for task_idx, task_length in enumerate(self.lengths):
            for local_idx in range(task_length):
                self.index_map.append((task_idx, local_idx))

        # 決定論的に全体をシャッフル
        rng = random.Random(42)
        rng.shuffle(self.index_map)

    def num_examples(self):
        return self.num_conversations

    def get_example(self, index):
        """
        すべてのサンプルを決定論的にシャッフルし、対話データを取得する
        データセットのサイズに関係なく、タスクが訓練全体にわたって混ぜ合わさる
        """

        # 入力を検証
        assert 0 <= index < self.num_conversations, f"Index {index} out of range for mixture with {self.num_conversations} conversations"

        # インデックスを取得
        task_idx, local_idx = self.index_map[index]

        # 対応するタスクから対話データを取得して返す
        return self.tasks[task_idx][local_idx]

mixed_task = TaskMixture(
    tasks=[smoltalk, mmlu, gsm, simple_spelling, spelling_bee],
    start=0,
    stop=5,
)
mixed_task.lengths, mixed_task.num_examples(), mixed_task[0]

### データセット

In [ ]:
# 中間学習用のデータセットを作成

base_dir = get_base_dir()

identity_conversations_filepath = os.path.join(
    base_dir, "identity_conversations.jsonl"
)

train_dataset = TaskMixture([
    # 46万件（460K）の一般的な会話データ
    SmolTalk(split="train"),

    # 10万件（100K）の多肢選択問題データ
    MMLU(subset="auxiliary_train", split="train"),

    # 8千件（8K）の数学問題データ
    GSM8K(subset="main", split="train"),

    # 1000件の合成された対話データ
    CustomJSON(filepath=identity_conversations_filepath),

    # 1000件の合成された対話データ（2エポック分）
    CustomJSON(filepath=identity_conversations_filepath),

    # 20万件の簡単なスペリングデータ
    SimpleSpelling(size=200000, split="train"),

    # 8万件のスペリングビー（文字数を数える）データ
    SpellingBee(size=80000, split="train"),
]) # total: 460K + 100K + 8K + 200K + 80K = 848K rows

train_dataset.lengths, train_dataset.num_examples()
# 合計 849,648件


In [ ]:
# 検証データセットを作成

val_dataset = TaskMixture([
    # 2万4千件（24K）の一般的な会話データ
    SmolTalk(split="test"),

    # 1万4千件（14K）の多肢選択問題データ
    MMLU(subset="all", split="test", stop=5200),

    # 1.32千件（1.32K）の数学問題データ
    GSM8K(subset="main", split="test", stop=420),
]) # total: 24K + 14K + 1.32K ~= 39K rows

val_dataset.lengths, val_dataset.num_examples()
# 合計 20,849件

In [ ]:
# データセットの最後に到達したかどうかを示すフラグ
last_step = False

# 進捗（0.0から1.0までの範囲）
approx_progress = 0.0

def mid_data_generator(split):
    """
    中間学習用のデータジェネレーター
    """

    # 1) ジェネレータの初期化

    # ジェネレータの状態を管理するグローバル変数を定義
    global last_step, approx_progress

    # 引数を検証
    assert split in {"train", "val"}, "split must be 'train' or 'val'"

    # データセットを選択
    dataset = train_dataset if split == "train" else val_dataset

    # データセットサイズを取得
    dataset_size = len(dataset)

    # データセットサイズを検証
    assert dataset_size > 0

    # 1バッチ分の合計トークン数を計算（+1は正解データのズレのため）
    needed_tokens = device_batch_size * max_seq_len + 1

    # バッファを作成
    token_buffer = deque()

    # 1バッチ分のトークンのテンソルを初期化
    scratch = torch.empty(
        needed_tokens,
        dtype=torch.int64,
        pin_memory=(device_type == "cuda") # ピンメモリでGPUへの転送を高速化
    )

    # 分散学習用のランクを基準としたデータセットのカーソルを初期化
    # プロセスごとに異なるドキュメントを処理するため
    cursor = ddp_rank

    # イテレーションカウンタを初期化
    it = 0

    # 2) ジェネレータループ

    while True:

        # 2-1) バッファが満たされるまでデータを追加

        while len(token_buffer) < needed_tokens:

            # 対話データを取得
            conversation = dataset[cursor]

            # 対話データをトークン化してIDを取得
            ids, _ = tokenizer.render_conversation(conversation)

            # トークンIDをバッファに追加
            token_buffer.extend(ids)

            # カーソルを更新
            cursor += ddp_world_size

            # カーソルがデータセットサイズを超えた場合
            if cursor >= dataset_size:
                # 最初に戻る
                cursor -= dataset_size

                # 訓練データの場合
                if split == "train":
                    # 最終ステップのフラグを有効化し終了
                    last_step = True
                    logger.info("ジェネレータを終了（データセットの最後に到達したため）")

        it += 1

        # 指定されたイテレーション数に達した場合
        if num_iterations > 0 and it >= num_iterations:

            if not last_step:
                logger.info(f"ジェネレータを終了（イテレーション数 {num_iterations} に達したため）")

            # 最終ステップのフラグを有効化し終了
            last_step = True

        # 2-2) 入力と正解データを構築

        # バッファから必要なトークン数をscratchにコピー
        for i in range(needed_tokens):
            scratch[i] = token_buffer.popleft()

        # 入力データ
        inputs_cpu = scratch[:-1].to(dtype=torch.int32)

        # 正解データ
        targets_cpu = scratch[1:]

        # 入力データをデバイスに転送（non_blocking=Trueで非同期転送）
        inputs = inputs_cpu.view(
            device_batch_size, max_seq_len
        ).to(device=device, dtype=torch.int32, non_blocking=True)

        # 正解データをデバイスに転送（non_blocking=Trueで非同期転送）
        targets = targets_cpu.view(
            device_batch_size, max_seq_len
        ).to(device=device, dtype=torch.int64, non_blocking=True)

        # 2-3) 進捗の更新

        if split == "train":
            if num_iterations > 0:
                # イテレーション数に基づいて進捗を計算
                approx_progress = it / num_iterations
            else:
                # データセットのカーソルに基づいて進捗を計算
                approx_progress = cursor / dataset_size

        # 2-4) データを供給

        yield inputs, targets


# 訓練データローダーを作成
train_loader = mid_data_generator("train")

# 検証データローダーを作成する関数
build_val_loader = lambda: mid_data_generator("val")

# 検証
x, y = next(train_loader)
x.shape, y.shape
# (8, 2048), (8, 2048)

### 最適化関数

In [ ]:
# 学習率のスケジューラを作成

# スケジューラで使用する進捗グローバル変数
# 0から1までの範囲で変化
progress = 0

def get_lr_multiplier(progress):
    """
    訓練進捗に応じて学習率を調整する関数

    Args:
        progress (float):
            訓練の進捗（0.0から1.0までの範囲）
    Returns:
        float:
            学習率の乗数
    """
    # 最初の80%の訓練では学習率を維持し、その後線形に0まで減衰させる
    return 1 if progress < 0.8 else 1 - (progress - 0.8) / 0.2

get_lr_multiplier(0.0), get_lr_multiplier(0.9), get_lr_multiplier(1.0)

In [ ]:
# Muon最適化関数のモーメンタムスケジューラ

def get_muon_momentum(it):
    frac = min(it / 300, 1)
    momentum = (1 - frac) * 0.85 + frac * 0.95
    return momentum

get_muon_momentum(0), get_muon_momentum(150), get_muon_momentum(300), get_muon_momentum(600)

### 評価関数

In [ ]:
@torch.no_grad()
def evaluate_bpb(model, batches, steps, token_bytes):
    """
    BPB (bits per byte)でモデルを評価する損失関数
    合計損失と合計バイト数を計算し、それらを割り算して正規化する
    """

    # 1) 初期化

    # 損失を記録するテンソルを初期化
    total_nats = torch.tensor(0.0, dtype=torch.float32, device=model.get_device())

    # モデルが予測したトークンの合計バイト数を記録するテンソルを初期化
    total_bytes = torch.tensor(0, dtype=torch.int64, device=model.get_device())

    # 検証データローダーのイテレーターを作成
    batch_iter = iter(batches)

    # 2) 評価ループ

    # 指定されたステップ数だけ評価を実行
    for _ in range(steps):

        # 入力シーケンスと正解シーケンスのバッチを取得
        x, y = next(batch_iter)

        # モデルの順伝搬を実行し、各トークンの損失を計算
        # loss_reduction='none'により、各トークンの損失が返される
        # (B, T)
        loss2d = model(x, y, loss_reduction='none')

        # フラット化
        # (B*T,)
        loss2d = loss2d.view(-1)

        # 正解トークンをフラット化
        # (B*T,)
        y = y.view(-1)

        # 正解シーケンスにignore_index（-1）が含まれている場合、それを除いて集計
        if (y.int() < 0).any():

            # マスクを作成
            valid = y >= 0

            # 無効なインデックスを0に置き換え
            y_safe = torch.where(valid, y, torch.zeros_like(y))

            # マスクされていないトークンをバイト数に変換
            num_bytes2d = torch.where(
                valid,
                token_bytes[y_safe],
                torch.zeros_like(y, dtype=token_bytes.dtype)
            )

            # ignore_indexと特殊トークンを除外して（num_bytes2d > 0）、損失を集計
            total_nats += (loss2d * (num_bytes2d > 0)).sum()

            # バイト数を集計
            total_bytes += num_bytes2d.sum()

        # 正解シーケンスにignore_indexが含まれていない場合、普通に集計
        else:
            # トークンをバイト数に変換
            num_bytes2d = token_bytes[y]

            # 特殊トークンを除外して（num_bytes2d > 0）、損失を集計
            total_nats += (loss2d * (num_bytes2d > 0)).sum()

            # バイト数を集計
            total_bytes += num_bytes2d.sum()


    # DDPが有効な場合、全プロセスで合計を集約
    world_size = dist.get_world_size() if dist.is_initialized() else 1
    if world_size > 1:
        dist.all_reduce(total_nats, op=dist.ReduceOp.SUM)
        dist.all_reduce(total_bytes, op=dist.ReduceOp.SUM)

    # 最終的なBPBを計算して返す
    total_nats = total_nats.item()
    total_bytes = total_bytes.item()
    if total_bytes == 0:
        return float('inf')
    
    # BPB = 総損失 / ln(2) / 総バイト数
    bpb = total_nats / (math.log(2) * total_bytes)
    return bpb

### 訓練

In [ ]:
def save_checkpoint(checkpoint_dir, step, model_data, optimizer_data, meta_data, rank=0):
    """
    モデルとオプティマイザのチェックポイントを保存するユーティリティ関数

    Args:
        checkpoint_dir (str):
            チェックポイントを保存するディレクトリのパス
        step (int):
            現在の訓練ステップ数
        model_data (dict):
            モデルの状態辞書
        optimizer_data (dict):
            オプティマイザの状態辞書
        meta_data (dict):
            メタデータ辞書（例: 訓練ステップ数、BPBなど）
        rank (int):
            分散学習におけるプロセスのランク（デフォルトは0）
    Returns:
        None
    """

    # 1) チェックポイントとメタデータの保存

    # ランク0の場合
    if rank == 0:

        # チェックポイントディレクトリを作成
        os.makedirs(checkpoint_dir, exist_ok=True)

        # モデルのチェックポイントを保存
        model_path = os.path.join(checkpoint_dir, f"model_{step:06d}.pt")
        torch.save(model_data, model_path)
        logger.info(f"Saved model parameters to: {model_path}")

        # メタデータをJSONとして保存
        meta_path = os.path.join(checkpoint_dir, f"meta_{step:06d}.json")
        with open(meta_path, "w", encoding="utf-8") as f:
            json.dump(meta_data, f, indent=2)

        logger.info(f"Saved metadata to: {meta_path}")

    # 2) 最適化関数の状態を保存

    if optimizer_data is not None:
        # パスを作成
        optimizer_path = os.path.join(checkpoint_dir, f"optim_{step:06d}_rank{rank:d}.pt")

        # 最適化関数の状態を保存
        torch.save(optimizer_data, optimizer_path)

        logger.info(f"Saved optimizer state to: {optimizer_path}")

In [ ]:
# 中間学習開始

# 1) 初期化

logger.setLevel(logging.INFO)
logger.info(f"中間学習を開始 {num_iterations=} {device_batch_size=} {grad_accum_steps=} {eval_every=}")

# データセットの最後に到達したかを示すフラグを初期化
last_step = False

# 進捗（0.0から1.0までの範囲）を初期化
approx_progress = 0.0

# 訓練データローダーを初期化
train_loader = mid_data_generator("train")

# バイトごとの平均損失（BPB）を初期化 
min_val_bpb = float("inf")

# 訓練損失の指数移動平均を初期化
smooth_train_loss = 0

# 指数移動平均の減衰係数
ema_beta = 0.9

# 訓練時間を初期化
total_training_time = 0

# ステップ数を初期化
step = 0

# 2) 訓練ループ

while True:

    # 2-1) 並列学習時の終了フラグの集約

    # 並列学習が有効な場合
    # False
    if ddp:
        logger.info("並列学習時の終了フラグを集約")

        # データセットの最後に到達したかを示すフラグを回収
        last_step_tensor = torch.tensor(
            last_step, dtype=torch.int32, device=device
        )
        dist.all_reduce(last_step_tensor, op=dist.ReduceOp.MAX)

        # いずれかのプロセスが最後のステップに到達した場合、終了フラグを有効化
        last_step = bool(last_step_tensor.item())

    # これまでの浮動小数点演算数（FLOPs）を計算
    flops_so_far = num_flops_per_token * total_batch_size * step

    # 2) バイトごとの平均損失（BPB）を計算

    # 評価インターバル、もしくは最後のステップで検証を実行
    if eval_every > 0 and (last_step or step % eval_every == 0) and step > 0:
        logger.info(f"バイトごとの平均損失（BPB）を計算 {step=:05d}")

        # 評価モードに変更
        model.eval()

        # 検証データローダーを作成
        val_loader = build_val_loader()

        # 評価に使用するトークン数からステップ数を計算
        eval_steps = eval_tokens // (
            device_batch_size * max_seq_len * ddp_world_size
        )

        # バイトごとの平均損失（BPB）を計算
        with autocast_ctx: # 自動混合精度（AMP）を有効化
            val_bpb = evaluate_bpb(
                model,
                val_loader, # 中間学習の訓練と同様のデータジェネレータを使用
                eval_steps,
                token_bytes
            )

        # ログ出力
        logger.info(f"Step {step:05d} | Validation bpb: {val_bpb:.4f} | total training time: {total_training_time/60:.2f}m | total training flops: {flops_so_far/1e15:.2f} PFLOPs")

        # 損失が改善した場合、最小値を更新
        if val_bpb < min_val_bpb:
            min_val_bpb = val_bpb

        # WandBにログを送信
        wandb_run.log({
            "step": step,
            "total_training_flops": flops_so_far,
            "total_training_time": total_training_time,
            "val/bpb": val_bpb,
        })

        # 訓練モードに戻す
        model.train()

    # 3) チェックポイントの保存

    # 最後のステップの場合
    if master_process and last_step and not dry_run:
        logger.info(f"チェックポイントを保存")

        # 出力ディレクトリ名を作成（d20など）
        output_dirname = f"d{depth}"

        # 出力先のパスを作成
        checkpoint_dir = os.path.join(
            base_dir, "mid_checkpoints", output_dirname
        )

        # チェックポイントを保存
        save_checkpoint(
            checkpoint_dir,
            step,
            orig_model.state_dict(),
            [opt.state_dict() for opt in optimizers],
            {
                "step": step,
                "val_bpb": val_bpb,
                "model_config": {
                    "sequence_len": max_seq_len,
                    "vocab_size": tokenizer.get_vocab_size(),
                    "n_layer": depth,
                    "n_head": model.config.n_head,
                    "n_kv_head": model.config.n_kv_head,
                    "n_embd": model.config.n_embd,
                },
                "user_config": user_config,
            }
        )

    # 4) 終了判定

    if last_step:
        break

    # 5) 訓練フェイズ

    # 5-1) 初期化

    # プロセスを同期
    synchronize()

    logger.info(f"訓練開始 {step=:05d}")

    # タイマーを開始
    t0 = time.time()

    # 5-2) 勾配蓄積のループ

    for micro_step in range(grad_accum_steps):
        logger.info(f"勾配蓄積ステップ {micro_step+1}/{grad_accum_steps}")

        # 自動混合精度（AMP）を有効化して順伝搬を実行
        with autocast_ctx:
            # 予測したロジットを使って、クロスエントロピー損失を計算
            loss = model(x, y)

        # ログ用に訓練損失を保存
        train_loss = loss.detach()

        # 勾配蓄積のために損失を正規化
        loss = loss / grad_accum_steps

        # 逆伝播を実行
        loss.backward()

        # 順伝播・逆伝播中に次のバッチを事前に取得
        x, y = next(train_loader)

        # 現在の進捗を更新
        progress = max(progress, approx_progress)

    # 5-3) 最適化関数とパラメータの更新

    # 訓練進捗から学習率の乗数を計算
    lrm = get_lr_multiplier(progress)

    # 全ての最適化関数の学習率を更新
    for opt in optimizers:
        for group in opt.param_groups:
            group["lr"] = group["initial_lr"] * lrm

    # Muon最適化関数のモーメンタムを進捗から計算
    muon_momentum = get_muon_momentum(step)

    # Muon最適化関数のモーメンタムを更新
    for group in muon_optimizer.param_groups:
        group["momentum"] = muon_momentum

    # パラメータを更新
    for opt in optimizers:
        opt.step()

    # 勾配をリセット
    model.zero_grad(set_to_none=True)

    # 全てのプロセスを同期
    synchronize()

    # 5-4) ロギング

    # タイマーを停止
    t1 = time.time()

    # 経過時間を計算
    dt = t1 - t0

    # ステップ数をインクリメント
    step += 1

    # 訓練損失の指数移動平均（EMA）を更新
    smooth_train_loss = ema_beta * smooth_train_loss \
        + (1 - ema_beta) * train_loss.item()

    # 訓練損失の指数移動平均（EMA）のバイアス補正
    debiased_smooth_loss = smooth_train_loss / (1 - ema_beta**(step + 1))

    # 進捗をパーセンテージで計算
    pct_done = 100 * progress

    # 毎秒あたりの処理トークン数を計算
    tok_per_sec = int(world_tokens_per_fwdbwd / dt)

    # 毎秒あたりの浮動小数点演算数（FLOPs）を計算
    flops_per_sec = num_flops_per_token * total_batch_size / dt

    # H100の理論上の最大FLOPsを計算
    # bfloat16 H100 SXMの場合、989 TFLOPs
    promised_flops_per_sec_h100 = 989e12 * ddp_world_size

    # GPUの使用率（MFU）を計算
    # 単位はパーセンテージ
    mfu = 100 * flops_per_sec / promised_flops_per_sec_h100

    # 総訓練時間を更新
    # コンパイル時間を除外するため、最初の10ステップはカウントしない
    if step > 10:
        total_training_time += dt

    # ログ出力
    logger.info(f"step {step:05d} ({pct_done:.2f}%) | loss: {debiased_smooth_loss:.6f} | lrm: {lrm:.2f} | dt: {dt * 1000:.2f}ms | tok/sec: {tok_per_sec:,} | mfu: {mfu:.2f} | total time: {total_training_time/60:.2f}m")

    # 10ステップごとにWandBにログを送信
    if step % 10 == 0:
        wandb_run.log({
            "step": step,
            "total_training_flops": flops_so_far,
            "total_training_time": total_training_time,
            "train/loss": debiased_smooth_loss,
            "train/lrm": lrm,
            "train/dt": dt,
            "train/tok_per_sec": tok_per_sec,
            "train/mfu": mfu,
        })

logger.setLevel(logging.DEBUG)

## SFT（教師有りファインチューニング）

SFTは「アシスタントの回答のみ」を使って、モデルが適切な回答を生成する能力を訓練する

訓練に必要なデータ数は中間学習の1/40程度

科学推論（ARC）や数学（GSM8K）などの正解が明確なタスクが中心

In [ ]:
import os
import wandb
import torch
import torch.distributed as dist
from contextlib import nullcontext

### ハイパーパラメータ

In [ ]:
# 1) 実行環境とログ設定

# WandBの実行名
# dummyの場合はWandBにログを送信しない
run = "dummy"

# SFTで使用するモデル
# base（ベースモデル）かmid（中間学習済みモデル）
source = "mid"

# 使用するモデルのタグ
# Noneの場合、最大のモデルが使用される
model_tag = None

# 使用するモデルのステップ数
# Noneの場合、最新のモデルが使用される
step = None

In [ ]:
# 2) 計算リソースと精度

# デバイスタイプ
# cuda、cpu、mps（空文字列の場合は自動検出）
device_type = ""

# データ型
dtype = "bfloat16"

# 1GPUあたりのバッチサイズ
# SFTでは会話データが長くなるため小さめに設定
# デフォルトは4
device_batch_size = 1

In [ ]:
# 3) 学習期間とバッチサイズ

# 訓練のエポック数
# 過学習を防ぐため1エポックで十分
num_epochs = 10

# 総ステップ数
# -1の場合、エポック数から自動計算
num_iterations = 10

# 1ステップあたりの目標サンプル数
# device_batch_sizeが小さくても勾配蓄積で補うため
# -1の場合は無効化し、num_epochsから自動計算
target_examples_per_step = 32

In [ ]:
# 4) 最適化関数の設定

# 最終層の学習率
unembedding_lr = 0.004

# 埋め込み層の学習率
embedding_lr = 0.2

# Transformerの内部の行列用の学習率
matrix_lr = 0.02

# 重み減衰（L2正則化）の強さ
weight_decay = 0.0

# 初期学習率の割合（学習率スケジューラ用）
init_lr_frac = 0.02

In [ ]:
# 5) 評価設定

# 評価の間隔
eval_every = 100

# 評価のステップ数
eval_steps = 100

# 評価のメトリクスの間隔
eval_metrics_every = 200

# 評価で使用する最大の問題数
eval_metrics_max_problems = 1024

In [ ]:
# 6) ログ用

user_config = {k: globals()[k] for k in config_keys}
user_config

### パラメータ自動設定

In [ ]:
# デバイスを自動検出
device_type = autodetect_device_type() \
    if device_type == "" else device_type
logger.info(f"{device_type=}")

# 分散学習の設定を取得
ddp, ddp_rank, ddp_local_rank, ddp_world_size, device = compute_init(device_type)

# マスタープロセスのフラグ
master_process = ddp_rank == 0

# 設定のデータ型をPyTorchのデータ型に変換
ptdtype = torch.float32 if dtype == 'float32' else torch.bfloat16

# 自動混合精度（AMP）のコンテキストマネージャーを作成
autocast_ctx = torch.amp.autocast(device_type=device_type, dtype=ptdtype) \
    if device_type == "cuda" else nullcontext()

In [ ]:
examples_per_step = device_batch_size * ddp_world_size
assert target_examples_per_step % examples_per_step == 0, "Target examples per step must be divisible by examples per step"
logger.info(f"ステップあたりのサンプル数 {examples_per_step=}")

grad_accum_steps = target_examples_per_step // examples_per_step
logger.info(f"勾配蓄積ステップ数 {grad_accum_steps=}")

In [ ]:
# イテレーション数を自動計算の場合

if num_iterations == -1:
    # 検証
    assert num_epochs > 0, "num_epochs must be positive if num_iterations is -1"

    # エポック数からイテレーション数を計算
    num_iterations = (len(train_ds) // target_examples_per_step) * num_epochs
    logger.info(f"イテレーション数を自動計算 {num_iterations=}")
else:
    logger.info(f"イテレーション数 {num_iterations=}")

### WandBの設定

In [ ]:
# WandBの設定

# ダミーのWandBクラスを使用するか
use_dummy_wandb = run == "dummy" or not master_process
logger.info(f"{use_dummy_wandb=}")

# WandBの初期化
wandb_run = DummyWandb() \
    if use_dummy_wandb \
    else wandb.init(
        project="nanochat-sft",
        name=run,
        config=user_config,
        save_code=True
    )

### モデルを読み込み

In [ ]:
def find_largest_model(checkpoint_dir):
    """
    チェックポイントディレクトリから最大のモデルタグを見つけて返す
    """

    # 1) モデルのタグを取得

    model_tags = [
        f for f in os.listdir(checkpoint_dir) \
            if os.path.isdir(os.path.join(checkpoint_dir, f))
    ]

    if not model_tags:
        raise FileNotFoundError(f"No checkpoints found in {checkpoint_dir}")

    # 2) 候補を見つける（d29などdから始まるもの）

    candidates = []

    for model_tag in model_tags:
        match = re.match(r"d(\d+)", model_tag)
        if match:
            model_depth = int(match.group(1))
            candidates.append((model_depth, model_tag))

    if candidates:
        candidates.sort(key=lambda x: x[0], reverse=True)
        return candidates[0][1]

    # 3) 候補が見つからなかった場合、最新のモデルを返す
    model_tags.sort(
        key=lambda x: os.path.getmtime(os.path.join(checkpoint_dir, x)),
        reverse=True
    )

    return model_tags[0]

# 検証
find_largest_model(get_base_dir() + "/mid_checkpoints")

In [ ]:
def find_last_step(checkpoint_dir):
    """
    チェックポイントディレクトリから最新のモデルを見つけて、そのステップ数を返す
    """

    # 1) チェックポイントファイルを取得
    checkpoint_files = glob.glob(os.path.join(checkpoint_dir, "model_*.pt"))

    if not checkpoint_files:
        raise FileNotFoundError(f"No checkpoints found in {checkpoint_dir}")

    # 2) 最新のステップ数を抽出

    last_step = int(
        max(
            os.path.basename(f).split("_")[-1].split(".")[0] \
                for f in checkpoint_files
        )
    )

    return last_step

# 検証

find_last_step(get_base_dir() + "/mid_checkpoints/d20")

In [ ]:
# モデルを読み込み
model, tokenizer, meta = load_model(
    source,
    device,
    phase="train",
    model_tag=model_tag,
    step=step
)

# コンパイル前のモデルを保存
# エクスポート時に層の名前を参照するため
orig_model = model

# SFTの場合は入力の長さが変わりコンパイルが頻繁に発生するため、コメントアウト
# model = torch.compile(model, dynamic=True)

# 評価用に推論エンジンを初期化
engine = Engine(model, tokenizer)

### タスク

#### ARC

In [ ]:
class ARC(Task):
    """
    AI2（Allen Institute for AI）が提供するARCデータセットの多肢選択問題タスク
    内容は小学校高学年から中学生レベルの科学問題
    ARC-Challengeは2590問
    ARC-Easyは5200問
    id, question, choices, answerKeyのフィールドから成る
    https://huggingface.co/datasets/allenai/ai2_arc
    """

    def __init__(self, subset, split, **kwargs):
        super().__init__(**kwargs)

        # 1) 入力を検証

        assert subset in ["ARC-Easy", "ARC-Challenge"], "ARC subset must be ARC-Easy or ARC-Challenge"
        assert split in ["train", "validation", "test"], "ARC split must be train|validation|test"

        # 2) データセットを読み込み

        self.ds = load_dataset("allenai/ai2_arc", subset, split=split).shuffle(seed=42)

    @property
    def eval_type(self):
        return 'categorical'

    def num_examples(self):
        return len(self.ds)

    def get_example(self, index):

        # 1) インデックスからデータを抽出
        row = self.ds[index]
        question = row["question"]
        choices = row["choices"]["text"]
        letters = row["choices"]["label"]
        answer_string = row["answerKey"]

        # 正解を検証
        assert answer_string in letters, f"ARC answer {answer_string} must be one of {letters}"

        # 2) 対話データを構築

        # 多肢選択問題のフォーマットでメッセージをレンダリング
        user_message = render_mc(question, letters, choices)

        # メッセージを構築
        messages = [
            {"role": "user", "content": user_message},
            {"role": "assistant", "content": answer_string}
        ]

        # 整形
        conversation = {
            "messages": messages,
            "letters": letters, # 評価時に使用
        }

        return conversation

    def evaluate(self, conversation, assistant_response):
        """
        多肢選択問題の正解を評価
        """

        # 回答に選択肢の文字が含まれていることを検証
        assert assistant_response in conversation['letters'], f"ARC answer {assistant_response} is expected to be one of {conversation['letters']}"

        # 正解のメッセージを取得
        assistant_message = conversation['messages'][-1]['content']

        # 正誤判定
        return assistant_response == assistant_message

arc = ARC(subset="ARC-Easy", split="validation")
arc.num_examples(), arc[0]

#### HumanEval

In [ ]:
def extract_imports(prompt):
    """
    プロンプトのコードブロックからインポート文を抽出する

    Args:
        prompt (str): コードブロックを含むプロンプト
    Returns:
        str: 抽出されたインポート文の文字列
    """

    imports = []

    for line in prompt.split('\n'):
        stripped = line.strip()

        if stripped.startswith('import ') \
            or stripped.startswith('from '):

            imports.append(stripped)

        elif stripped and not stripped.startswith('#'):
            # Stop at first non-import, non-comment line
            break

    return '\n'.join(imports)

# 検証

extract_imports("""
import os
import sys

# This is a comment
from math import sqrt

def foo():
    pass
""")

In [ ]:
def extract_program(completion):
    """
    LLMの出力からPythonコードを抽出する

    様々な形式の出力に対応:
    - ```python ... ``` または ``` ... ``` ブロックで囲まれたコード
    - マークダウンブロックなしのプレーンコード
    - コードブロックの前後に余分なテキストがあるコード

    Args:
        completion (str): LLMの出力文字列
    Returns:
        str: 抽出されたPythonコード。見つからない場合は元の出力を返す。
    """

    # マークダウンコードブロックの正規表現パターン
    # ```python\n...\n``` または ```\n...\n``` にマッチ
    pattern = r'```(?:python)?\s*\n(.*?)\n```'

    # 正規表現でコードブロックを検索
    matches = re.findall(pattern, completion, re.DOTALL)

    if matches:
        # 最初に見つかったコードブロックを返す
        return matches[0].strip()

    # コードブロックが見つからなかった場合は元の出力を返す
    return completion.strip()

# 検証

extract_program("""
Here is a sample code:

```python
import os
import sys
print("Hello, World!")
```

That's all!
""")

In [ ]:
class HumanEval(Task):
    """
    OpenAIのHumanEvalデータセットのコード生成タスク
    Pythonのコーディング能力を評価するために使用
    """

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        # データセットを読み込み
        self.ds = load_dataset(
            "openai/openai_humaneval",
            split="test"
        ).shuffle(seed=42)

    @property
    def eval_type(self):
        return 'generative'

    def num_examples(self):
        return len(self.ds)

    def get_example(self, index):
        """
        データセットから一つの問題を取得する
        """

        # データを抽出
        row = self.ds[index]
        prompt = row['prompt']
        solution = row['canonical_solution']
        entry_point = row['entry_point']
        test = row['test']

        # 正解をプロンプトに追加
        complete_solution = f"{prompt}\n{solution}"

        # メッセージを構築
        messages = [
            {"role": "user", "content": prompt},
            {"role": "assistant", "content": complete_solution},
        ]

        # 整形
        conversation = {
            "messages": messages,
            "entry_point": entry_point, # 評価時に使用
            "test": test, # 評価時に使用
        }

        return conversation

    def evaluate(self, conversation, completion):
        """
        モデルが生成したコード（completion）を実行して正誤を判定する

        Args:
            conversation (dict):
                対話データ。get_example()の出力と同じ形式
            completion (str):
                モデルが生成したコード
        Returns:
            bool: 正解ならTrue、誤りならFalse
        """

        # 対話データからインポート文を抽出
        # モデルの出力にはインポート文が含まれないことが多いため
        imports = extract_imports(conversation['messages'][0]['content'])

        # モデルの出力からコードを抽出
        completion_code = extract_program(completion)

        # テストコードを組み立て
        program = (
            imports
            + "\n\n"
            + completion_code
            + "\n\n"
            + conversation['test']
            + "\n"
            + f"check({conversation['entry_point']})"
        )

        # コードを実行
        result = execute_code(program)

        # 正誤を返す
        success = result.success
        return success

human_eval = HumanEval()
human_eval.num_examples(), human_eval[0]

### データセット

In [ ]:
# 訓練データセットを作成

# 合成対話データのパスを取得
identity_conversations_filepath = os.path.join(
    get_base_dir(), "identity_conversations.jsonl"
)

# 複数のタスクを混ぜてデータセットを構築
train_ds = TaskMixture([
    # 2300件のARC-Easyデータ
    ARC(subset="ARC-Easy", split="train"),

    # 1100件のARC-Challengeデータ
    ARC(subset="ARC-Challenge", split="train"),

    # 8千件のGSM8Kデータ
    GSM8K(subset="main", split="train"),

    # 1万件のSmolTalkデータ
    SmolTalk(split="train", stop=10_000),

    # 1千件の合成対話データ
    CustomJSON(filepath=identity_conversations_filepath),

    # 300件のSimple Spellingデータ
    SimpleSpelling(size=300, split="train"),

    # 300件のSpelling Beeデータ
    SpellingBee(size=300, split="train"),
])

train_ds.lengths, train_ds.num_examples()
# 合計 22,439件

In [ ]:
# 検証用のデータセットを作成

# 2万4千件のSmolTalkデータ（一部のみ使用）
val_ds = SmolTalk(split="test")

In [ ]:
def sft_data_generator(dataset, batch_size):
    """
    中間学習用のデータジェネレータ
    データセットをバッチにまとめ、可変長の対話データを固定長のテンソルに変換して供給する

    Args:
        dataset (TaskMixture):
            対話データセット
        batch_size (int):
            バッチサイズ
    Yields:
        Tuple[torch.Tensor, torch.Tensor]:
            入力シーケンスと正解シーケンスのバッチ
    """

    # パディングトークンIDを取得（損失計算時にマスクされるため問題ない）
    pad_token_id = tokenizer.encode_special("<|assistant_end|>")

    def collate_and_yield(batch):
        """
        可変長の対話データを固定長の正方形テンソルに変換する
        """

        # 1) 初期化

        # バッチサイズを取得
        nrows = len(batch)

        # 最大のシーケンス長を取得
        # 1トークンを予測するために1つ短くする
        ncols = max(len(ids) for ids, mask in batch) - 1

        # 入力のテンソルをパッドトークンで初期化
        inputs = torch.full((nrows, ncols), pad_token_id, dtype=torch.long)

        # 正解のテンソルを-1で初期化
        # -1はignore_indexに対応
        targets = torch.full((nrows, ncols), -1, dtype=torch.long)

        # 2) データを各テンソルにコピー

        # バッチ内の各対話データを各テンソルにコピー
        for i, (ids, mask) in enumerate(batch):

            # シーケンス長を取得
            n = len(ids)

            # トークンIDのテンソルを作成
            ids_tensor = torch.tensor(ids, dtype=torch.long)

            # 最後のトークンを除いて入力テンソルにコピー
            inputs[i, :n-1] = ids_tensor[:-1]

            # 正解テンソルを作成
            row_targets = ids_tensor[1:]

            # 最初のトークン（BOS）を除いて正解のマスクのテンソルを作成
            mask_tensor = torch.tensor(mask[1:], dtype=torch.long)

            # ユーザーの発言部分（User=0, Assistant=1）を-1に上書き
            # ユーザーの発言を予測しないようにするため
            row_targets[mask_tensor == 0] = -1

            # 正解テンソルにコピー
            targets[i, :n-1] = row_targets

        # 3) デバイスに転送して返す

        inputs = inputs.to(device)
        targets = targets.to(device)

        return inputs, targets

    # メインループ

    # バッチを初期化
    batch = []

    while True:

        # 並列学習時は使用するデータを重複しないようにループ
        for i in range(ddp_rank, len(dataset), ddp_world_size):

            # データを取得
            doc = dataset[i]

            # 対話データをトークンIDのシーケンスに変換
            ids, mask = tokenizer.render_conversation(doc)

            # バッチに追加
            batch.append((ids, mask))

            # データ数がバッチサイズに達した場合
            if len(batch) == batch_size:

                # バッチを整形して供給
                yield collate_and_yield(batch)

                # バッチを初期化
                batch = []

# 訓練データローダーを作成
train_loader = sft_data_generator(train_ds, batch_size=device_batch_size)

# 検証データローダーを作成する関数
build_val_loader = lambda: sft_data_generator(val_ds, batch_size=device_batch_size)

x, y = next(train_loader)
x.shape, y.shape

### 最適化関数

In [ ]:
# AdamWとMuon最適化関数を初期化

optimizers = model.setup_optimizers(
    unembedding_lr=unembedding_lr,
    embedding_lr=embedding_lr,
    matrix_lr=matrix_lr,
    weight_decay=weight_decay,
)
optimizers

In [ ]:
# 初期学習率を設定

for opt in optimizers:
    for group in opt.param_groups:
        # 学習率に初期学習率の割合を乗算
        group["lr"] = group["lr"] * init_lr_frac

        # 初期学習率を保存
        # 進捗率から学習率を計算するために使用
        group["initial_lr"] = group["lr"]

In [ ]:
# 線形減衰スケジューラ用のヘルパー関数を作成

def get_lr_multiplier(it):
    """
    イテレーション数に基づいて学習率を線形に減衰させる

    Args:
        it (int): 現在のイテレーション数
    Returns:
        float: 学習率の乗数
    """
    lrm = 1.0 - it / num_iterations
    return lrm

get_lr_multiplier(0), get_lr_multiplier(num_iterations // 2), get_lr_multiplier(num_iterations)

### 評価

In [ ]:
def run_generative_eval(task_object, tokenizer, model, engine, num_samples, max_new_tokens, temperature, top_k, max_problems=None):
    """
    汎用的な生成タスクを評価する（ジェネレーティブ評価）

    プロンプトに対して続きを生成し、タスクオブジェクトのevaluate()メソッドで正誤を判定する

    - GSM8K（数学）
    - HumanEval（コード生成）
    - Spelling Bee（スペルチェック）
    - Simple Spelling（スペルチェック）

    Args:
        task_object (Task): 評価するタスクオブジェクト
        tokenizer (Tokenizer): トークナイザー
        model (Model): 言語モデル
        engine (Engine): 推論エンジン
        num_samples (int): 各問題に対して生成するサンプル数
        max_new_tokens (int): 生成する最大トークン数
        temperature (float): 生成の温度パラメータ
        top_k (int): top-kサンプリングのk値
        max_problems (int, optional): 評価する最大の問題数。Noneの場合、全ての問題を評価する
    Returns:
        float: 正解率（accuracy）
    """

    # 1) 分散学習の設定

    # 分散学習の情報を取得
    ddp, ddp_rank, ddp_local_rank, ddp_world_size = get_dist_info()

    # デバイスを取得
    device = model.get_device()

    # 2) 評価

    # 評価する問題数を決定
    num_problems = len(task_object) \
        if max_problems is None else min(len(task_object), max_problems)

    # 正解数とループ数を初期化
    num_passed, total = 0, 0

    # 各プロセスが担当する問題をループ
    for i in range(ddp_rank, num_problems, ddp_world_size):

        # 対話データを取得
        conversation = task_object[i]

        # プロンプトをトークン化
        encoded_prompt = tokenizer.render_for_completion(conversation)

        # 最大トークン数までテキスト生成
        results, _ = engine.generate_batch(
            encoded_prompt,
            num_samples=num_samples,
            max_tokens=max_new_tokens,
            temperature=temperature,
            top_k=top_k,
        )

        # 生成箇所より前のトークンの長さを取得
        prefix_length = len(encoded_prompt)

        # 生成された部分を抽出してデコード
        completions = [
            tokenizer.decode(result_tokens[prefix_length:]) \
                for result_tokens in results
        ]

        # 生成された部分を評価
        outcomes = [
            task_object.evaluate(conversation, completion) \
                for completion in completions \
            ]

        # 正解数をカウント
        passed = any(outcomes)

        # ループ回数をインクリメント
        total += 1

        # 合計正解数を更新
        num_passed += int(passed)

        # ログ
        logger.info(f"\r\033[KRank {ddp_rank} | {num_passed}/{total} ({100*num_passed/total:.2f}%)")

    # 並列学習が有効な場合、結果を集約
    if ddp:
        num_passed_tensor = torch.tensor([num_passed], dtype=torch.long, device=device)
        total_tensor = torch.tensor([total], dtype=torch.long, device=device)
        dist.all_reduce(num_passed_tensor, op=dist.ReduceOp.SUM)
        dist.all_reduce(total_tensor, op=dist.ReduceOp.SUM)
        num_passed = num_passed_tensor.item()
        total = total_tensor.item()

    logger.info("=" * 50)
    logger.info(f"Final: {num_passed}/{total} ({100*num_passed/total:.2f}%)")

    # 正解率を返す
    return num_passed/total

In [ ]:
def run_categorical_eval(task_object, tokenizer, model, batch_size, max_problems=None):
    """
    多肢選択タスクを評価する（カテゴリカル評価）

    カテゴリカルタスクは、多肢選択問題のように、あらかじめ定義された選択肢から正解を選ぶタスク
    選択肢のトークンのロジットを比較して最も高いものを選択する
    - ARC（多肢選択問題）
    - MMLU（多肢選択問題）

    Args:
        task_object (Task): 評価するタスクオブジェクト
        tokenizer (Tokenizer): トークナイザー
        model (Model): 言語モデル
        batch_size (int): バッチサイズ
        max_problems (int, optional): 評価する最大の問題数。Noneの場合、全ての問題を評価する
    Returns:
        float: 正解率（accuracy）
    """

    # 1) 分散学習の設定

    ddp, ddp_rank, ddp_local_rank, ddp_world_size = get_dist_info()

    device = model.get_device()

    # BOSをパッドトークンとして使用（これらの位置は無視されるため問題ない）
    bos = tokenizer.get_bos_token_id()

    # 問題数を決定
    num_problems = len(task_object) \
        if max_problems is None else min(len(task_object), max_problems)

    # バッチの数を計算
    ceil_div = lambda x, y: -(-x // y)
    num_batches = ceil_div(num_problems, batch_size)

    # 2) 評価ループ

    # IDキャッシュを初期化
    # これは多くの文字が頻繁に繰り返されるため、トークナイザーの作業を節約するためのもの
    letter_to_id_cache = {}

    # 正解数とループ数を初期化
    num_passed, total = 0, 0

    # 各プロセスが担当するバッチをループ
    for i in range(ddp_rank, num_batches, ddp_world_size):

        # 2-1) バッチの準備

        # バッチの開始と終了のインデックスを計算
        i0, i1 = i * batch_size, min((i + 1) * batch_size, num_problems)

        # 問題のバッチを取得
        conversations = [task_object[ii] for ii in range(i0, i1)]

        # プロンプトをトークンIDに変換
        prompt_ids = [
            tokenizer.render_for_completion(conversation) \
                for conversation in conversations
        ]

        # バッチ内の最大長を取得
        max_length = max(len(ids) for ids in prompt_ids)

        # 各対話データの回答開始位置を取得
        answer_time_positions = [len(ids) - 1 for ids in prompt_ids]

        # 各対話データを最大長にパディング
        padded_prompt_ids = [
            ids + [bos] * (max_length - len(ids)) for ids in prompt_ids
        ]

        # テンソルに変換してデバイスに転送
        prompt_ids = torch.tensor(padded_prompt_ids, dtype=torch.long, device=device)

        # 2-2) モデルで推論

        # 勾配計算を無効化
        with torch.no_grad():
            # モデルで順伝播してロジットを取得
            logits = model(prompt_ids) # (B, T, V)

        # 2-3) バッチ内の各問題を評価

        for idx, conversation in enumerate(conversations):

            # 全ての選択肢の文字のトークンIDを取得
            letters = conversation['letters']

            letter_ids = []
            for letter in letters:

                # キャッシュにない場合
                if not letter in letter_to_id_cache:
                    # トークンIDを取得してキャッシュに保存
                    encoded_letter = tokenizer.encode(letter)
                    assert len(encoded_letter) == 1, "Each letter must be a single token"
                    letter_to_id_cache[letter] = encoded_letter[0]

                # キャッシュからトークンIDを取得
                letter_ids.append(letter_to_id_cache[letter])

            # 回答位置を取得
            answer_pos = answer_time_positions[idx]

            # 回答位置での選択肢の文字のロジットを抽出
            focus_logits = logits[idx, answer_pos, letter_ids]

            # 最も高いロジットを持つ文字のIDを取得
            argmax_letter_id = focus_logits.argmax(dim=-1).item()

            # 予測した文字を取得
            predicted_letter = letters[argmax_letter_id]

            # 結果を評価
            outcome = task_object.evaluate(conversation, predicted_letter)

            # 合計正解数を更新
            num_passed += int(outcome)

            # ループ回数をインクリメント
            total += 1

    # 並列学習が有効な場合、結果を集約
    if ddp:
        num_passed_tensor = torch.tensor([num_passed], dtype=torch.long, device=device)
        total_tensor = torch.tensor([total], dtype=torch.long, device=device)
        dist.all_reduce(num_passed_tensor, op=dist.ReduceOp.SUM)
        dist.all_reduce(total_tensor, op=dist.ReduceOp.SUM)
        num_passed = num_passed_tensor.item()
        total = total_tensor.item()

    # 3) 平均を計算
    average = num_passed/total

    logger.info(f"Final: {num_passed}/{total} ({100*average:.2f}%)")
    return average

In [ ]:
def run_chat_eval(task_name, model, tokenizer, engine, batch_size=1, num_samples=1, max_new_tokens=512, temperature=0.0, top_k=50, max_problems=None):
    """
    指定されたタスクでチャットモデルを評価する
    run_generative_eval()またはrun_categorical_eval()のラッパー

    Args:
        task_name (str): タスク名（HumanEvalなど）
        model (Model): 言語モデル
        tokenizer (Tokenizer): トークナイザー
        engine (Engine): 推論エンジン
        batch_size (int): バッチサイズ（カテゴリカル評価の場合に使用）
        num_samples (int): 各問題に対して生成するサンプル数（生成評価の場合に使用）
        max_new_tokens (int): 生成する最大トークン数（生成評価の場合に使用）
        temperature (float): 生成の温度パラメータ（生成評価の場合に使用）
        top_k (int): top-kサンプリングのk値（生成評価の場合に使用）
        max_problems (int, optional): 評価する最大の問題数。Noneの場合、全ての問題を評価する
    Returns:
        float: 正解率（accuracy）
    """

    # 1) タスクオブジェクトの作成

    # 対応するタスクモジュールを選択
    task_module = {
        'HumanEval': HumanEval,
        'MMLU': partial(MMLU, subset="all", split="test"),
        'ARC-Easy': partial(ARC, subset="ARC-Easy", split="test"),
        'ARC-Challenge': partial(ARC, subset="ARC-Challenge", split="test"),
        'GSM8K': partial(GSM8K, subset="main", split="test"),
        'SpellingBee': partial(SpellingBee, size=256, split="test"),
    }[task_name]

    # インスタンス化
    task_object = task_module()

    # 2) 評価を実行

    # 生成タスクの場合
    if task_object.eval_type == 'generative':
        acc = run_generative_eval(
            task_object,
            tokenizer,
            model,
            engine,
            num_samples,
            max_new_tokens,
            temperature,
            top_k,
            max_problems=max_problems
        )

    # 多肢選択タスクの場合
    elif task_object.eval_type == 'categorical':
        acc = run_categorical_eval(
            task_object,
            tokenizer,
            model,
            batch_size,
            max_problems=max_problems
        )
    else:
        raise ValueError(f"Unsupported task evaluation type: {task_object.eval_type}")

    return acc

### 訓練

In [ ]:
logger.setLevel(logging.INFO)

# 1) 初期化

# ステップ数を初期化
step = 0

# 訓練データローダーを初期化
train_loader = sft_data_generator(train_ds, batch_size=device_batch_size)

# ジェネレータをイテレート可能にする
train_iter = iter(train_loader)

# メトリクスの初期化
metrics = {}

# 2) 訓練ループ

for step in range(num_iterations):

    # 最終ステップかどうかを判定
    last_step = step == num_iterations - 1

    # 2-1) 検証データセットで評価

    if last_step or step % eval_every == 0:
        logger.info(f"検証損失を評価開始")
        model.eval()

        # 検証データローダーを初期化し、イテレータを作成
        val_iter = iter(build_val_loader())
        
        # 検証損失を初期化
        losses = []

        # 評価のステップ数分だけループ
        for _ in range(eval_steps):

            # 入力と正解を取得
            val_inputs, val_targets = next(val_iter)

            # 勾配計算を無効化し、自動混合精度コンテキストを使用
            with torch.no_grad(), autocast_ctx:
                # モデルで順伝播して損失を計算
                loss = model(val_inputs, val_targets)

            # 検証損失を保存
            losses.append(loss)

        # 検証損失の平均を計算
        val_loss = torch.stack(losses).mean()

        # 分散学習の場合
        if ddp:
            # 全プロセスで平均を計算
            dist.all_reduce(val_loss, op=dist.ReduceOp.AVG)

        # テンソルの検証損失をPythonの数値に変換
        val_loss = val_loss.item()

        logger.info(f"Step {step:05d} | Validation loss: {val_loss:.6f}")

        # WandBにログを送信
        wandb_run.log({
            "step": step,
            "val_loss": val_loss,
        })

        # モデルを訓練モードに戻す
        model.train()

    # 2-2) チャットタスクの評価

    if last_step or (step > 0 and step % eval_metrics_every == 0):
        logger.info(f"チャットタスクで評価開始")

        # 評価モードに変更
        model.eval()

        # 勾配計算を無効化し、自動混合精度コンテキストを使用（これにより2倍のバッチサイズが可能）
        with torch.no_grad(), autocast_ctx:

            # テキストを生成し、MMLUの正解率を計算
            metrics["mmlu_acc"] = run_chat_eval(
                "MMLU",
                model,
                tokenizer,
                engine,
                batch_size=device_batch_size*2,
                max_problems=eval_metrics_max_problems
            )

            # テキストを生成し、ARC-Easyの正解率を計算
            metrics["arc_easy_acc"] = run_chat_eval(
                "ARC-Easy",
                model,
                tokenizer,
                engine,
                batch_size=device_batch_size*2,
                max_problems=eval_metrics_max_problems
            )

        # ログ出力
        metrics_str = ', '.join(f'{k}: {v:.6f}' for k, v in metrics.items())
        logger.info(f"Step {step:05d} | {metrics_str}")

        # WandBにログを送信
        wandb_run.log({
            "step": step,
            **metrics,
        })

        # モデルを訓練モードに戻す
        model.train()

    # 2-3) 終了判定

    if last_step:
        break

    # 2-4) 訓練フェイズ

    # 訓練に使用したトークン数をゼロで初期化（active tokens）
    num_tokens = torch.tensor(0, device=device)

    # 勾配蓄積ステップのループ
    for micro_step in range(grad_accum_steps):
        logger.info(f"勾配蓄積ステップ {micro_step+1}/{grad_accum_steps} を処理中 {step=}")

        # 入力と正解を取得
        train_inputs, train_targets = next(train_iter)

        # 自動混合精度コンテキストを使用
        with autocast_ctx:
            # モデルでロジットを予測して、クロスエントロピー損失を計算（NSP）
            loss = model(train_inputs, train_targets)

        # 訓練損失の値を取得
        train_loss = loss.detach()

        # 損失を勾配蓄積数で正規化
        loss = loss / grad_accum_steps

        # 誤差逆伝播
        loss.backward()

        # 使用したトークン数を更新
        num_tokens += (train_targets >= 0).sum()

    # 分散学習の場合、全プロセスでトークン数を集約
    if ddp:
        dist.all_reduce(num_tokens, op=dist.ReduceOp.SUM)

    # 2-5) 学習率の更新

    # 学習率の乗数を取得
    lrm = get_lr_multiplier(step)

    # 各最適化関数の学習率を更新
    for opt in optimizers:
        for group in opt.param_groups:
            group["lr"] = group["initial_lr"] * lrm

    # モデルのパラメータを更新
    for opt in optimizers:
        opt.step()

    # 勾配をクリア
    model.zero_grad(set_to_none=True)

    # 2-6) ログ出力

    # 訓練損失のテンソルをPythonの数値に変換
    train_loss_item = train_loss.item()

    # 使用したトークン数をPythonの数値に変換
    num_tokens_item = num_tokens.item()

    # ログ出力
    logger.info(f"Step {step:05d}/{num_iterations:05d} | Training loss: {train_loss_item:.6f}| lrm: {lrm:.6f}| num_tokens: {num_tokens_item:,}")

    # WandBにログを送信
    wandb_run.log({
        "step": step,
        "lrm": lrm,
        "train_loss": train_loss_item,
        "num_tokens": num_tokens_item,
    })

    # ステップ数をインクリメント
    step += 1

logger.setLevel(logging.DEBUG)

In [ ]:
# モデルを保存
if master_process:
    # ベースディレクトリを取得
    base_dir = get_base_dir()

    # モデルの層数を取得
    depth = model.config.n_layer

    # 層数に応じたモデルタグを作成
    model_tag = f"d{depth}"

    # チェックポイントディレクトリを作成
    checkpoint_dir = os.path.join(base_dir, "chatsft_checkpoints", model_tag)

    # GPTConfigのキーワード引数を取得
    model_config_kwargs = model.config.__dict__

    # モデルを保存
    save_checkpoint(
        checkpoint_dir,
        step,
        model.state_dict(),
        None, # 最適化関数の状態は保存しない
        {
            "step": step,
            "val_loss": val_loss,
            **metrics,
            "model_config": model_config_kwargs,
        }
    )
    logger.info(f"モデルを保存しました {checkpoint_dir}")

In [ ]:
# WandBの終了
wandb_run.finish()

def compute_cleanup():
    """
    並列学習環境のクリーンアップ
    """
    if is_ddp():
        dist.destroy_process_group()

compute_cleanup()

## 強化学習

簡易的な[GRPO（Group Relative Policy Optimization）][1]を使用して強化学習:

1. GSM8K（数学問題）をモデルに入力し、異なる回答を16個生成（ロールアウト）
2. 回答を正誤判定し、正解に1.0の報酬、誤りに0.0の報酬を与える
3. 平均を計算し、各回答のアドバンテージ（良い回答は正で悪い回答は負）を計算（GRPO）
4. 生成した回答をモデルに通し、回答ごとのクロスエントロピー損失（負の対数尤度）を計算
5. 対数尤度とアドバンテージを掛けて、方策勾配法の目的関数を計算
6. 目的関数の符号を反転した値を損失とみなし、誤差逆伝播を実行
7. 最適化関数でモデルのパラメータを更新

[1]: https://arxiv.org/pdf/2402.03300

In [ ]:
import os
import itertools
import re
import wandb
import torch
import torch.distributed as dist

### ハイパーパラメータ

In [ ]:
# 1) 実行環境の設定

# WandBの実行名
run = "dummy"

# 使用するモデル
# mid（中間学習済みモデル）またはsft（微調整済みモデル）
source = "sft" # mid|sft

# 使用する計算精度
dtype = "bfloat16"

In [ ]:
# 2) 強化学習のサンプリングの設定

# 1GPUあたりの最大バッチサイズ
# デフォルトは8
device_batch_size = 1

# 1回の更新ステップで扱う問題の総数
examples_per_step = 16

# 1つの問題に対して生成する回答の数
num_samples = 16

In [ ]:
# 3) 生成パラメータ 

# 最大生成トークン数
max_new_tokens = 256

# 生成の温度パラメータ
temperature = 1.0

# 確率上位50個の単語からサンプリング
top_k = 50

In [ ]:
# 4) 最適化関数の設定

# RLは学習が不安定なため慎重な設定

# 出力層の学習率
unembedding_lr = 0.004

# 埋め込み層の学習率
embedding_lr = 0.2

# 行列パラメータの学習率
matrix_lr = 0.02

# 重み減衰
weight_decay = 0.0

# 初期学習率の係数（5%から開始）
init_lr_frac = 0.05

In [ ]:
# 5) 訓練と評価設定

# エポック数
num_epochs = 1

# モデルを保存するインターバル
save_every = 60

# pass@k（k回生成して正解が含まれる確率）評価のインターバル
eval_every = 60

# 評価で使用する問題数
eval_examples = 400

In [ ]:
# 6) ログ用

config_keys = [k for k,v in globals().items() if not k.startswith('_') and isinstance(v, (int, float, bool, str))]
user_config = {k: globals()[k] for k in config_keys} # will be useful for logging
user_config

### パラメータ自動設定

In [ ]:
# 分散学習の初期化
ddp, ddp_rank, ddp_local_rank, ddp_world_size, device = compute_init()
logger.info(f"{ddp=}, {ddp_rank=}, {ddp_local_rank=}, {ddp_world_size=}, {device=}")

# マスタープロセスの判定
master_process = ddp_rank == 0
logger.info(f"{master_process=}")

# 計算精度の設定
dtype = torch.float32 if dtype == 'float32' else torch.bfloat16
logger.info(f"{dtype=}")

# 自動混合精度コンテキストを作成
autocast_ctx = torch.amp.autocast(device_type="cuda", dtype=dtype)

### WandBの設定

In [ ]:
# ダミーのWandBクラスを使用
use_dummy_wandb = run == "dummy" or not master_process
logger.info(f"{use_dummy_wandb=}")

# WandBの初期化
wandb_run = DummyWandb() \
    if use_dummy_wandb \
    else wandb.init(project="nanochat-rl", name=run, config=user_config)

### モデルを読み込み

In [ ]:
# モデル・トークナイザー・メタデータの読み込み
model, tokenizer, meta = load_model(source, device, phase="eval")

# 推論エンジンの初期化
engine = Engine(model, tokenizer)

### データセット

In [ ]:
# 訓練データセットを読み込み
# 7473問
train_task = GSM8K(subset="main", split="train")
logger.info(f"訓練データセットのサイズ: {len(train_task)}")

# 検証データセットを読み込み
# 1319問
val_task = GSM8K(subset="main", split="test")
logger.info(f"検証データセットのサイズ: {len(val_task)}")

# 総ステップ数を計算
num_steps = (len(train_task) // examples_per_step) * num_epochs
logger.info(f"訓練の総ステップ数: {num_steps}")
# 467問

In [ ]:
@torch.no_grad() # 勾配計算を無効化
def get_batch():
    """
    強化学習のためにロールアウト（テキスト生成）を実行し、アドバンテージを含む訓練データを生成するジェネレータ

    Yields:
        generated_token_sequences (list of list of int):
            各サンプルの生成されたトークンIDのリスト
        inputs (torch.LongTensor of shape (B, T)):
            モデルへの入力トークンIDのテンソル
        targets (torch.LongTensor of shape (B, T)):
            モデルの正解トークンIDのテンソル
        rewards (torch.FloatTensor of shape (B,)):
            各サンプルの報酬のテンソル
        advantages (torch.FloatTensor of shape (B,)):
            各サンプルのアドバンテージのテンソル
    """

    # 1) 初期化

    # パディング用の特殊トークンIDを取得（損失計算で無視されるので問題ない）
    assistant_end = tokenizer.encode_special("<|assistant_end|>")

    # 各プロセスが担当する訓練データのインデックスを計算
    rank_indices = range(ddp_rank, len(train_task), ddp_world_size)

    # 2) 訓練データのインデックスでループ

    for example_idx in itertools.cycle(rank_indices):

        # ユーザーとアシスタントの両方を含む対話データを取得
        # train_taskはGSM8Kタスクオブジェクトで小学生向けの数学問題が含まれる
        conversation = train_task[example_idx]

        # 正解を取り除き、<|assistant_start|>トークンまでを残してトークン化
        tokens = tokenizer.render_for_completion(conversation)

        # プロンプトの長さを取得
        prefix_length = len(tokens)

        # モデルを評価モードに設定
        model.eval()

        # 生成されたトークンIDのリストを初期化
        generated_token_sequences = []

        # 生成されたマスクのリストを初期化
        masks = []

        # 1回のバッチで生成するサンプル数を決定
        num_sampling_steps = num_samples // device_batch_size

        # 3) ロールアウト

        # 回答を生成するループ（メモリ不足を防ぐためにループ処理）
        for sampling_step in range(num_sampling_steps):

            # ループごとにシードを変更
            seed = hash((step, example_idx, sampling_step)) & 0x7FFFFFFF

            # 自動混合精度コンテキストを使用
            with autocast_ctx:
                # エンジンでテキストを生成
                generated_token_sequences_batch, masks_batch = engine.generate_batch(
                    tokens,
                    num_samples=device_batch_size,
                    max_tokens=max_new_tokens,
                    temperature=temperature,
                    top_k=top_k,
                    seed=seed,
                )

            # 生成したトークンIDをリストに追加
            generated_token_sequences.extend(generated_token_sequences_batch)

            # 生成したマスクをリストに追加
            masks.extend(masks_batch)

        # 4) 報酬の計算

        # 報酬のリストを初期化
        rewards = []

        # 生成したシーケンスごとにループ
        for sample_tokens in generated_token_sequences:
            # プロンプトの後の生成されたトークンのみを取得
            generated_tokens = sample_tokens[prefix_length:]

            # 生成された応答をデコード
            generated_text = tokenizer.decode(generated_tokens)

            # 対話データと生成されたテキストから報酬を計算
            reward = train_task.reward(conversation, generated_text)

            # 報酬をリストに追加
            rewards.append(reward)

        # 5) データの整形

        # 生成されたシーケンスの最大長を取得
        max_length = max(len(seq) for seq in generated_token_sequences)

        # 各シーケンスを最大長までパディング
        padded_generated_token_sequences = [
            seq + [assistant_end] * (max_length - len(seq)) \
                for seq in generated_token_sequences
        ]

        # 各マスクを最大長までパディング
        padded_masks = [
            mask + [0] * (max_length - len(mask)) \
                for mask in masks
        ]

        # 各シーケンスをPyTorchのテンソルに変換してデバイスに転送
        # (B, T)
        ids = torch.tensor(
            padded_generated_token_sequences,
            dtype=torch.long,
            device=device
        )

        # 各マスクをPyTorchのテンソルに変換してデバイスに転送
        # (B, T)
        mask_ids = torch.tensor(
            padded_masks,
            dtype=torch.long,
            device=device
        )

        # 入力を作成
        # (B, T-1)
        inputs = ids[:, :-1]

        # 正解を作成
        # 上書きを防ぐためクローン
        # (B, T-1)
        targets = ids[:, 1:].clone()

        # マスクが0の位置の正解を-1に設定（損失計算で無視されるようにする）
        # (B, T-1)
        targets[mask_ids[:, 1:] == 0] = -1

        # 報酬をPyTorchのテンソルに変換してデバイスに転送
        # (B,)
        rewards = torch.tensor(rewards, dtype=torch.float, device=device)

        # 報酬の平均を計算
        mu = rewards.mean()

        # アドバンテージを計算
        # GRPOの簡易版でグループ内の平均に対する差分がアドバンテージ
        # 本来は (x - mu) / sigma が望ましいが、平均のみを使用
        # (B,)
        advantages = rewards - mu

        # データを供給
        yield generated_token_sequences, inputs, targets, rewards, advantages

In [ ]:
def run_gsm8k_eval(task, tokenizer, engine,
    max_examples=None,
    num_samples=1,
    max_completion_tokens=256,
    temperature=0.0,
    top_k=50
):
    """
    GSM8Kタスク（小学生レベルの数学問題）をpass@k評価し、評価結果のレコードのリストを返す
    pass@kは、k回生成して正解が含まれる確率を測定する評価方法

    Args:
        task (GSM8K): GSM8Kタスクオブジェクト
        tokenizer (Tokenizer): トークナイザー
        engine (Engine): 推論エンジン
        max_examples (int, optional): 評価する最大の問題数。Noneの場合、全ての問題を評価する
        num_samples (int): 各問題に対して生成するサンプル数
        max_completion_tokens (int): 生成する最大トークン数
        temperature (float): 生成の温度パラメータ
        top_k (int): top-kサンプリングのk値
    Yields:
        record (dict): 各問題の評価結果のレコード
    """

    # 1) 分散学習の設定

    #　評価する問題数を決定
    max_examples = min(max_examples, len(task)) \
        if max_examples is not None else len(task)

    # 2) 評価ループ

    for idx in range(ddp_rank, max_examples, ddp_world_size):

        # 対話データを取得
        conversation = task[idx]

        # プロンプトをトークン化
        tokens = tokenizer.render_for_completion(conversation)

        # 生成箇所より前のトークンの長さを取得
        prefix_length = len(tokens)

        # 生成する問題数がバッチサイズ以下であることを確認
        assert num_samples <= device_batch_size
        
        # 回答をバッチ生成
        generated_token_sequences, masks = engine.generate_batch(
            tokens,
            num_samples=num_samples,
            max_tokens=max_completion_tokens,
            temperature=temperature, # 多様なパターンをサンプリング
            top_k=top_k
        )

        # 生成された各サンプルの正誤をチェック
        outcomes = []

        # 各生成シーケンスでループ
        for sample_tokens in generated_token_sequences:

            # 生成したトークン列からプロンプトを除去
            generated_tokens = sample_tokens[prefix_length:]

            # トークン列をデコード
            generated_text = tokenizer.decode(generated_tokens)

            # 正誤を評価
            is_correct = task.evaluate(conversation, generated_text)

            # 結果を保存
            outcomes.append({
                "is_correct": is_correct
            })

        # 結果を整形

        record = {
            "idx": idx,
            "outcomes": outcomes,
        }

        yield record

### 最適化関数

In [ ]:
# 最適化関数を初期化

optimizers = model.setup_optimizers(
    unembedding_lr=unembedding_lr,
    embedding_lr=embedding_lr,
    matrix_lr=matrix_lr,
    weight_decay=weight_decay,
)

optimizers

In [ ]:
# 最適化関数の学習率を設定

for opt in optimizers:
    for group in opt.param_groups:
        group["lr"] = group["lr"] * init_lr_frac
        group["initial_lr"] = group["lr"] # 減衰用に保存

In [ ]:
def get_lr_multiplier(it):
    """
    学習率の乗数を取得
    """
    lrm = 1.0 - it / num_steps
    return lrm

get_lr_multiplier(0), get_lr_multiplier(num_steps//2), get_lr_multiplier(num_steps-1)

In [ ]:
logger.info(f"ステップあたりの合計生成数: {examples_per_step * num_samples}")
assert examples_per_step % ddp_world_size == 0, "Desired examples per step must be divisible by the number of ranks"

examples_per_rank = examples_per_step // ddp_world_size
logger.info(f"各ランクあたりのステップあたりの問題数: {examples_per_rank}")

### 訓練

In [ ]:
logger.setLevel(logging.INFO)

batch_iterator = get_batch()

for step in range(num_steps):

    # 1) eval_everyごとにpass@k評価を実行

    if step % eval_every == 0:
        logger.info(f"GSM8Kで評価を実行中 {step=}")

        # 評価モードに変更
        model.eval()

        # pass@kの計算用のテンソルをゼロで初期化
        # 1つの問題につきdevice_batch_size個のサンプルを生成する
        passk = torch.zeros(device_batch_size, device=device)

        # 自動混合精度コンテキストを使用
        with autocast_ctx:

            # GSM8Kタスクで評価を実行
            # device_batch_size個のサンプルを生成
            records_iter = run_gsm8k_eval(
                val_task,
                tokenizer,
                engine,
                num_samples=device_batch_size,
                max_examples=eval_examples,
                temperature=1.0
            )

            # 評価結果をリストに変換
            records = list(records_iter)

        # 生成したk個の回答の中に少なくとも1つの正解があるかを集計
        # pass@1, pass@2, ..., pass@device_batch_sizeまでを評価
        for k in range(1, device_batch_size + 1):
            passk[k - 1] = sum(
                any(o["is_correct"] for o in r["outcomes"][:k]) for r in records
            )

        # 全レコード数を取得
        num_records = torch.tensor(len(records), dtype=torch.long, device=device)

        # 分散学習の場合、全ランクで集約
        if ddp:
            dist.all_reduce(num_records, op=dist.ReduceOp.SUM)
            dist.all_reduce(passk, op=dist.ReduceOp.SUM)

        # pass@kを生成数で正規化
        passk = passk / num_records.item()

        # ログ出力
        print_passk = [f"Pass@{k}: {passk[k - 1].item():.4f}" for k in range(1, device_batch_size + 1)]
        logger.info(f"Step {step} | {', '.join(print_passk)}")

        # WandBにログを送信
        log_passk = {f"pass@{k}": passk[k - 1].item() for k in range(1, device_batch_size + 1)}
        wandb_run.log({
            "step": step,
            **log_passk,
        })

    # 2) 訓練フェーズ

    # 報酬のリストを初期化
    rewards_list = []

    # シーケンス長のリストを初期化
    sequence_lengths = []

    # 各GPUが担当する問題数でループ
    for example_step in range(examples_per_rank):
        logger.info(f"訓練中 {step=}, {example_step=}/{examples_per_rank}")

        # 2-1) ロールアウト

        # 1つの問題に対するnum_samples個の訓練データを取得
        sequences_all, inputs_all, targets_all, rewards_all, advantages_all = next(batch_iterator)

        # 2-2) 勾配の計算

        # 訓練モードに変更
        model.train()

        # 訓練データ数がdevice_batch_sizeの倍数であることを確認
        assert inputs_all.size(0) % device_batch_size == 0

        # num_samples個の訓練データをdevice_batch_size個ずつに分割して処理
        num_passes = inputs_all.size(0) // device_batch_size

        # 分割ごとにループし、勾配を計算（勾配蓄積）
        for pass_idx in range(num_passes):
            logger.info(f"勾配計算中 {step=}, {example_step=}, pass {pass_idx+1}/{num_passes}")

            # 2-2-1) ミニバッチを取得

            # 開始と終了のインデックスを計算
            b0, b1 = pass_idx * device_batch_size, (pass_idx + 1) * device_batch_size
            inputs = inputs_all[b0:b1]
            targets = targets_all[b0:b1]
            rewards = rewards_all[b0:b1]
            advantages = advantages_all[b0:b1]

            # 2-2-2) 順伝播を実行

            # 自動混合精度を有効化
            with autocast_ctx:
                # モデルで順伝播して、クロスエントロピー損失を計算（=負の対数尤度）し、符号を反転（=対数尤度）
                # (B, T)
                logp = -model(inputs, targets, loss_reduction='none').view_as(inputs)

            # 2-2-3) 方策勾配法の目的関数を計算

            # J(\theta) = \sum{\log{p(a|s)} * A(s,a)}
            # 対数尤度にアドバンテージを乗じて和を計算
            # これにより良い結果（A>0)のトークンの確率を上げ、逆を下げる
            pg_obj = (logp * advantages.unsqueeze(-1)).sum()

            # 2-2-4) 目的関数の値を正規化

            # 有効なトークン数を計算（最低でも1を保証しゼロ除算を防ぐ）
            # targetsが-1の部分はプロンプト・ツール出力・パッディングのため損失計算対象ではない
            num_valid = (targets >= 0).sum().clamp(min=1)

            # 勾配蓄積用に正規化
            # 有効なトークン数、分割ループ数、担当する問題数の乗数で割って正規化
            pg_obj = pg_obj / (num_valid * num_passes * examples_per_rank)

            # 2-2-5) 損失を計算

            # 損失は目的関数の符号を反転したもの
            loss = -pg_obj

            # 2-2-6) 誤差逆伝播

            loss.backward()

            # ログ出力
            logger.info(f"Step {step}/{num_steps} | Example step {example_step} | Pass {pass_idx} | loss: {loss.item():.6f} | Average reward: {rewards.mean().item()}")

        # ログ用

        # 報酬の平均を計算してリストに追加 
        rewards_list.append(rewards_all.mean().item())

        # シーケンス長をリストに追加
        sequence_lengths.extend(len(seq) for seq in sequences_all)

    # 3) ログ出力

    # 平均報酬を計算
    mean_reward = sum(rewards_list) / len(rewards_list)

    # 平均シーケンス長を計算
    mean_sequence_length = sum(sequence_lengths) / len(sequence_lengths)

    # 分散学習の場合、全ランクで集約
    if ddp:
        # 平均報酬のテンソルを作成
        mean_reward_tensor = torch.tensor(
            mean_reward, dtype=torch.float, device=device
        )

        # 平均シーケンス長のテンソルを作成
        mean_sequence_length_tensor = torch.tensor(
            mean_sequence_length, dtype=torch.float, device=device
        )

        # 全ランクで平均報酬を集約
        dist.all_reduce(mean_reward_tensor, op=dist.ReduceOp.AVG)

        # 全ランクで平均シーケンス長を集約
        dist.all_reduce(mean_sequence_length_tensor, op=dist.ReduceOp.AVG)

        # 集約した平均報酬をPythonの数値に変換
        mean_reward = mean_reward_tensor.item()

        # 集約した平均シーケンス長をPythonの数値に変換
        mean_sequence_length = mean_sequence_length_tensor.item()

    logger.info(f"Step {step}/{num_steps} | Average reward: {mean_reward} | Average sequence length: {mean_sequence_length:.2f}")

    # WandBにログを送信
    wandb_run.log({
        "step": step,
        "reward": mean_reward,
        "sequence_length": mean_sequence_length,
    })

    # 4) 最適化関数を更新

    lrm = get_lr_multiplier(step)
    for opt in optimizers:
        for group in opt.param_groups:
            group["lr"] = group["initial_lr"] * lrm

    # 5) パラメータを更新
    for opt in optimizers:
        opt.step()

    # 6) 後処理

    # 勾配をクリア
    model.zero_grad(set_to_none=True)

    # WandBに学習率の乗数を送信
    wandb_run.log({
        "step": step,
        "lrm": lrm,
    })

    # 7) モデルを保存

    # マスタープロセスでステップ数がsave_everyの場合
    if master_process and \
        ((step > 0 and step % save_every == 0) or step == num_steps - 1):

        # ベースディレクトリを取得
        base_dir = get_base_dir()

        # モデルの層数を取得
        depth = model.config.n_layer

        # 層数に応じたモデルタグを作成
        model_tag = f"d{depth}"

        # チェックポイントディレクトリを作成
        checkpoint_dir = os.path.join(base_dir, "chatrl_checkpoints", model_tag)

        # GPTConfigのキーワード引数を取得
        model_config_kwargs = model.config.__dict__

        # モデルを保存
        save_checkpoint(
            checkpoint_dir,
            step,
            model.state_dict(),
            None, # 最適化関数の状態は保存しない
            {
                "model_config": model_config_kwargs,
            }
        )
        logger.info(f"モデルを保存しました {checkpoint_dir}")

logger.setLevel(logging.DEBUG)